# 부산 지역 카페/음식점 평점 분석
## Part 1: 데이터 로드, 전처리, 탐색적 자료분석(EDA)

**대주제:** 부산 지역 카페/음식점의 평점은 가격, 위치, 리뷰 수, 업종에 따라  
어떻게 달라지며, 이러한 요인들의 설명력은 어디까지인가?

**데이터 출처:** Google Places API (New)  
**수집 URL:** https://developers.google.com/maps/documentation/places/web-service  
**수집 날짜:** 2026-03-17

---
## 0. 환경 설정

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

# ── 한글 폰트 설정 (환경에 맞게 수정) ──
# Windows: "Malgun Gothic"
# Mac: "AppleGothic"  
# Linux: "Noto Sans CJK KR"
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 120

# 시각화 스타일
sns.set_style("whitegrid")
sns.set_palette("Set2")

print("환경 설정 완료")

---
## 1. 데이터 로드

Google Places API (New)를 통해 부산 16개 행정구의 음식점과 카페 데이터를 수집하였다.  
API에서 제공하는 필드 중 평점(rating), 리뷰 수(review_count), 가격대(price_level),  
업종(business_type), 주소(address) 등을 활용한다.

In [ ]:
# 원본 데이터 로드
df_raw = pd.read_csv("google_places_20260317_090717.csv")

print(f"원본 데이터: {df_raw.shape[0]}행 × {df_raw.shape[1]}열")
print(f"\n컬럼 목록:")
for i, col in enumerate(df_raw.columns, 1):
    print(f"  {i:2d}. {col:<20s} (dtype: {df_raw[col].dtype})")

In [ ]:
# 상위 5행 확인
df_raw.head()

---
## 2. 전처리

### 2.1 분석 대상 선정 기준

본 연구에서는 **가격대(price_level)** 정보가 존재하는 가게만을 분석 대상으로 선정하였다.  
그 이유는 다음과 같다:

1. 가격은 본 연구의 핵심 독립변수 중 하나이므로, 가격 정보가 없으면 소주제 1(가격대별 평점)과  
   소주제 5(통합 모델)에서 해당 관측치를 활용할 수 없다.
2. 가격대 정보가 있는 가게는 Google에서 충분한 정보가 축적된 가게이므로,  
   평점과 리뷰 수의 신뢰도가 상대적으로 높다.
3. 분석 전체에서 동일한 데이터셋을 사용함으로써 소주제 간 일관성을 확보한다.

또한, 부산광역시의 공식 16개 행정구·군에 해당하지 않는 관측치는 제외하였다.

In [ ]:
# ── 2.1 분석 대상 필터링 ──

# 부산 16개 행정구·군
valid_districts = [
    "해운대구", "부산진구", "금정구", "남구", "수영구",
    "중구", "동래구", "사하구", "북구", "사상구",
    "연제구", "영도구", "강서구", "동구", "서구", "기장군"
]

# 조건: (1) price_level 존재 (2) rating 존재 (3) 유효 행정구
df = df_raw[
    df_raw["price_level"].notna() &
    df_raw["rating"].notna() &
    df_raw["district"].isin(valid_districts)
].copy()

print(f"필터링 결과:")
print(f"  원본: {len(df_raw)}개")
print(f"  → price_level 존재 + rating 존재 + 유효 행정구: {len(df)}개")
print(f"  제거된 행: {len(df_raw) - len(df)}개")

### 2.2 변수 확인 및 정리

In [ ]:
# ── 2.2 결측치 확인 ──
print("필터링 후 결측치 현황:")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "  → 주요 분석 변수에 결측치 없음")

print(f"\n각 변수별 유효 데이터 수:")
print(f"  rating:       {df['rating'].notna().sum()}")
print(f"  review_count: {df['review_count'].notna().sum()}")
print(f"  price_level:  {df['price_level'].notna().sum()}")
print(f"  district:     {df['district'].notna().sum()}")
print(f"  business_type:{df['business_type'].notna().sum()}")

### 2.3 변수 변환이 필요한 이유 확인

변수를 변환하기 **전에**, 현재 데이터가 어떤 상태인지 먼저 시각적으로 확인한다.

In [ ]:
# ── 2.3-1 price_level 원본 상태 확인 ──
# "왜 범주화가 필요한가?"를 데이터로 보여주기

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# (좌) price_level 원본 분포 — 문제점이 보인다
price_raw_counts = df["price_level"].value_counts().sort_index()
labels_raw = {1.0: "1\n(INEXPENSIVE)", 2.0: "2\n(MODERATE)", 3.0: "3\n(EXPENSIVE)", 4.0: "4\n(VERY_EXPENSIVE)"}
colors_raw = ["#70AD47", "#5B9BD5", "#ED7D31", "#C00000"]
bars = axes[0].bar([labels_raw[k] for k in price_raw_counts.index], price_raw_counts.values, 
                    color=colors_raw, edgecolor="white")
for bar, val in zip(bars, price_raw_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 10, f"{val}개\n({val/len(df)*100:.1f}%)", 
                 ha="center", fontsize=10, fontweight="bold")
axes[0].set_ylabel("가게 수", fontsize=11)
axes[0].set_title("(a) price_level 원본 분포\n→ 4(VERY_EXPENSIVE)가 5개뿐", fontsize=12)

# (우) 범주화 후 — 고가(3) + 최고가(4)를 합침
# 먼저 범주화 수행
def categorize_price(level):
    if level == 1:
        return "저가"
    elif level == 2:
        return "중가"
    else:  # 3, 4
        return "고가"

df["price_category"] = df["price_level"].apply(categorize_price)
price_cat_counts = df["price_category"].value_counts().reindex(["저가", "중가", "고가"])
colors_cat = ["#70AD47", "#5B9BD5", "#ED7D31"]
bars = axes[1].bar(price_cat_counts.index, price_cat_counts.values, color=colors_cat, edgecolor="white")
for bar, val in zip(bars, price_cat_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 10, f"{val}개\n({val/len(df)*100:.1f}%)", 
                 ha="center", fontsize=10, fontweight="bold")
axes[1].set_ylabel("가게 수", fontsize=11)
axes[1].set_title("(b) 범주화 후: 고가 + 최고가 합침\n→ 3그룹으로 비교 가능", fontsize=12)

plt.suptitle("price_level 변환: 원본 → 범주화", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── 2.3-1b Class Imbalance 수치 비교 ──
# 합치기 전후로 클래스 불균형이 얼마나 개선되었는지 정량적으로 확인

n = len(df)

# 합치기 전 (4 클래스)
before_counts = df["price_level"].value_counts().sort_index()
props_before = before_counts / n
ir_before = before_counts.max() / before_counts.min()  # Imbalance Ratio
entropy_before = -np.sum(props_before * np.log(props_before))
balance_before = entropy_before / np.log(len(before_counts))  # 1이면 완전 균형

# 합친 후 (3 클래스)
after_counts = df["price_category"].value_counts().reindex(["저가", "중가", "고가"])
props_after = after_counts / n
ir_after = after_counts.max() / after_counts.min()
entropy_after = -np.sum(props_after * np.log(props_after))
balance_after = entropy_after / np.log(len(after_counts))

# 비교 테이블
print("Class Imbalance 비교 (합치기 전 vs 후)")
print(f"{'─'*65}")
print(f"{'지표':<30} {'합치기 전(4cls)':<18} {'합친 후(3cls)':<15}")
print(f"{'─'*65}")
print(f"{'클래스 수':<30} {len(before_counts):<18} {len(after_counts):<15}")
print(f"{'최소 클래스 크기':<30} {before_counts.min():<18} {after_counts.min():<15}")
print(f"{'최대/최소 비율 (IR)':<30} {ir_before:<18.1f} {ir_after:<15.1f}")
print(f"{'Balance Ratio (0~1)':<30} {balance_before:<18.4f} {balance_after:<15.4f}")
print(f"{'─'*65}")
print(f"  IR: {ir_before:.0f}:1 → {ir_after:.0f}:1 (↓ 개선)")
print(f"  Balance Ratio: {balance_before:.3f} → {balance_after:.3f} (↑ 개선)")

# 시각화: 비교 막대그래프
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

metrics = ["Imbalance Ratio\n(최대/최소, 낮을수록 균형)", "Balance Ratio\n(0~1, 높을수록 균형)"]
before_vals = [ir_before, balance_before]
after_vals = [ir_after, balance_after]

x = np.arange(len(metrics))
w = 0.3
bars1 = axes[0].bar(x - w/2, before_vals, w, label="합치기 전 (4cls)", color="#C44E52", alpha=0.8)
bars2 = axes[0].bar(x + w/2, after_vals, w, label="합친 후 (3cls)", color="#5B9BD5", alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics, fontsize=10)
axes[0].set_ylabel("값", fontsize=11)
axes[0].set_title("(a) 불균형 지표 비교", fontsize=12)
axes[0].legend()
# 값 표시
for bar in bars1:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
                 f"{bar.get_height():.1f}", ha="center", fontsize=9)
for bar in bars2:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
                 f"{bar.get_height():.1f}", ha="center", fontsize=9)

# 비율 파이차트 (합친 후)
axes[1].pie(after_counts.values, labels=[f"{k}\n({v}개, {v/n*100:.1f}%)" for k, v in after_counts.items()],
            colors=["#70AD47", "#5B9BD5", "#ED7D31"], autopct="", startangle=90, 
            wedgeprops=dict(edgecolor="white", linewidth=2))
axes[1].set_title("(b) 합친 후 비율 분포\n→ 중가 74.8% 여전히 지배적", fontsize=12)

plt.suptitle("Class Imbalance 분석", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"\n※ 합친 후에도 중가(74.8%)가 지배적인 class imbalance가 존재함")
print(f"  → 소주제 1 분석 시 비모수 검정(Kruskal-Wallis) 사용 및 해석에 주의 필요")

print(f"\n결정: VERY_EXPENSIVE(4)가 5개뿐이므로 EXPENSIVE(3)와 합쳐 '고가'로 범주화")
print(f"  저가 (1): {(df['price_category']=='저가').sum()}개")
print(f"  중가 (2): {(df['price_category']=='중가').sum()}개")
print(f"  고가 (3+4): {(df['price_category']=='고가').sum()}개")

In [ ]:
# ── 2.3-2 review_count 원본 상태 확인 + 로그 변환 ──
# "왜 로그 변환이 필요한가?"를 데이터로 보여주기

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# (좌) 원본 분포 — 극단적 right-skewed
axes[0].hist(df["review_count"], bins=50, color="#5B9BD5", edgecolor="white", alpha=0.85)
axes[0].axvline(df["review_count"].mean(), color="red", linestyle="--", 
                label=f'평균: {df["review_count"].mean():.0f}')
axes[0].axvline(df["review_count"].median(), color="orange", linestyle="--", 
                label=f'중앙값: {df["review_count"].median():.0f}')
axes[0].set_xlabel("리뷰 수", fontsize=11)
axes[0].set_ylabel("빈도", fontsize=11)
axes[0].set_title(f"(a) 원본 분포\n왜도(skewness) = {df['review_count'].skew():.2f}", fontsize=12)
axes[0].legend(fontsize=9)

# (중) 로그 변환 후 — 훨씬 대칭적
df["log_review_count"] = np.log1p(df["review_count"])
axes[1].hist(df["log_review_count"], bins=30, color="#70AD47", edgecolor="white", alpha=0.85)
axes[1].axvline(df["log_review_count"].mean(), color="red", linestyle="--", 
                label=f'평균: {df["log_review_count"].mean():.2f}')
axes[1].axvline(df["log_review_count"].median(), color="orange", linestyle="--", 
                label=f'중앙값: {df["log_review_count"].median():.2f}')
axes[1].set_xlabel("log(1 + 리뷰 수)", fontsize=11)
axes[1].set_ylabel("빈도", fontsize=11)
axes[1].set_title(f"(b) 로그 변환 후\n왜도(skewness) = {df['log_review_count'].skew():.2f}", fontsize=12)
axes[1].legend(fontsize=9)

# (우) Q-Q plot으로 정규성 비교
from scipy.stats import probplot
probplot(df["log_review_count"], dist="norm", plot=axes[2])
axes[2].set_title("(c) 로그 변환 후 Q-Q Plot\n→ 정규분포에 근접", fontsize=12)

plt.suptitle("review_count 변환: 원본 → log(1+x)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"변환 전: 왜도 = {df['review_count'].skew():.2f} (심한 right-skew)")
print(f"변환 후: 왜도 = {df['log_review_count'].skew():.2f} (대칭에 가까움)")
print(f"→ 회귀분석 등에서 log_review_count를 사용하면 모델 가정에 더 적합")

In [ ]:
# ── 2.3-3 리뷰 수 구간 변수 생성 ──
# 소주제 2에서 구간별 비교에 사용

df["review_group"] = pd.cut(
    df["review_count"],
    bins=[0, 50, 100, 300, 1000, np.inf],
    labels=["~50", "51~100", "101~300", "301~1000", "1000+"],
    include_lowest=True,
)

fig, ax = plt.subplots(figsize=(8, 4))
rg_counts = df["review_group"].value_counts().sort_index()
bars = ax.bar(rg_counts.index, rg_counts.values, color="#5B9BD5", edgecolor="white")
for bar, val in zip(bars, rg_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 5, str(val), ha="center", fontweight="bold")
ax.set_xlabel("리뷰 수 구간", fontsize=11)
ax.set_ylabel("가게 수", fontsize=11)
ax.set_title("리뷰 수 구간별 가게 수", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print("리뷰 수 구간별 분포:")
print(rg_counts.to_string())

### 2.4 독립변수 간 다중공선성 사전 확인

소주제 5에서 다중회귀 모델을 적합하기 전에,  
독립변수 간 강한 상관이 있는지 사전에 확인한다.  
강한 상관(|r| > 0.7)이 있으면 회귀 계수의 추정이 불안정해질 수 있다.

In [ ]:
# 수치형 독립변수 간 상관계수
indep_vars = ["price_level", "review_count", "log_review_count"]
indep_corr = df[indep_vars].corr()

print("독립변수 간 Pearson 상관계수:")
print(indep_corr.round(3).to_string())

print(f"\n주요 쌍별 상관:")
print(f"  price_level ↔ review_count:     r = {df['price_level'].corr(df['review_count']):.3f}")
print(f"  price_level ↔ log_review_count: r = {df['price_level'].corr(df['log_review_count']):.3f}")

# review_count와 log_review_count는 변환 관계이므로 당연히 높음 → 둘 중 하나만 모델에 사용
print(f"  review_count ↔ log_review_count: r = {df['review_count'].corr(df['log_review_count']):.3f} (변환 관계, 둘 중 하나만 사용)")

print(f"\n판단:")
print(f"  price_level ↔ log_review_count: |r| = {abs(df['price_level'].corr(df['log_review_count'])):.3f} < 0.3")
print(f"  → 독립변수 간 심각한 다중공선성 없음")
print(f"  → 소주제 5에서 VIF로 정식 진단 예정")

### 2.5 전처리 완료 요약

In [ ]:
# ── 최종 데이터 요약 ──
print("=" * 55)
print("전처리 완료 데이터 요약")
print("=" * 55)
print(f"  총 관측치:     {len(df)}개")
print(f"  행정구:        {df['district'].nunique()}개")
print(f"  업종:          음식점 {(df['business_type']=='음식점').sum()}개 / 카페 {(df['business_type']=='카페').sum()}개")
print(f"")
print(f"  [종속변수]")
print(f"  rating:        {df['rating'].min()} ~ {df['rating'].max()} (평균: {df['rating'].mean():.3f}, 중앙값: {df['rating'].median()})")
print(f"")
print(f"  [독립변수]")
print(f"  price_category: 저가 {(df['price_category']=='저가').sum()} / 중가 {(df['price_category']=='중가').sum()} / 고가 {(df['price_category']=='고가').sum()}")
print(f"  review_count:   {df['review_count'].min()} ~ {df['review_count'].max()} (중앙값: {df['review_count'].median():.0f})")
print(f"  district:       {df['district'].nunique()}개 행정구 (최소 {df['district'].value_counts().min()}개 ~ 최대 {df['district'].value_counts().max()}개)")
print(f"  business_type:  음식점 / 카페")

# ── 분석에 사용할 변수 정리 ──
print(f"\n분석에 사용할 변수:")
print(f"  종속변수: rating (평점, 1.0~5.0)")
print(f"  독립변수: price_category, log_review_count, district, business_type")
print(f"  파생변수: price_category, log_review_count, review_group")

---
## 3. 탐색적 자료분석 (EDA)

본격적인 소주제 분석에 앞서, 데이터의 전반적인 구조와 분포를 탐색한다.

### 3.1 종속변수: 평점(rating) 분포

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# (좌) 히스토그램
axes[0].hist(df["rating"], bins=25, color="#5B9BD5", edgecolor="white", alpha=0.85)
axes[0].axvline(df["rating"].mean(), color="red", linestyle="--", linewidth=1.5, 
                label=f'평균: {df["rating"].mean():.2f}')
axes[0].axvline(df["rating"].median(), color="orange", linestyle="--", linewidth=1.5, 
                label=f'중앙값: {df["rating"].median():.1f}')
axes[0].set_xlabel("평점", fontsize=12)
axes[0].set_ylabel("빈도", fontsize=12)
axes[0].set_title("(a) 평점 분포", fontsize=13)
axes[0].legend(fontsize=10)

# (우) 박스플롯
bp = axes[1].boxplot(df["rating"], vert=True, patch_artist=True, 
                      boxprops=dict(facecolor="#5B9BD5", alpha=0.7))
axes[1].set_ylabel("평점", fontsize=12)
axes[1].set_title("(b) 평점 박스플롯", fontsize=13)
axes[1].set_xticklabels(["전체"])

# 통계량 텍스트
stats_text = (f'n = {len(df)}\n'
              f'평균 = {df["rating"].mean():.2f}\n'
              f'표준편차 = {df["rating"].std():.2f}\n'
              f'왜도 = {df["rating"].skew():.2f}')
axes[1].text(1.3, df["rating"].min() + 0.1, stats_text, fontsize=10, 
             verticalalignment='bottom', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle("Figure 1. 종속변수(평점) 분포", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

**해석:**  
- 평점은 평균 4.14, 중앙값 4.2로 **좌편향(left-skewed)** 분포를 보인다.
- 대부분의 가게가 3.5~4.5 사이에 밀집되어 있으며, 2점대 이하는 극소수이다.
- 이는 온라인 평점의 일반적 특성(긍정 편향, 천장효과)과 일치한다.

### 3.2 독립변수 분포

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# (a) 가격대 분포
price_counts = df["price_category"].value_counts().reindex(["저가", "중가", "고가"])
colors_price = ["#70AD47", "#5B9BD5", "#ED7D31"]
bars = axes[0, 0].bar(price_counts.index, price_counts.values, color=colors_price, edgecolor="white")
for bar, val in zip(bars, price_counts.values):
    axes[0, 0].text(bar.get_x() + bar.get_width()/2, val + 10, str(val), ha="center", fontweight="bold")
axes[0, 0].set_ylabel("가게 수", fontsize=11)
axes[0, 0].set_title("(a) 가격대별 가게 수", fontsize=12)

# (b) 리뷰 수 분포 (로그 스케일)
axes[0, 1].hist(df["log_review_count"], bins=30, color="#5B9BD5", edgecolor="white", alpha=0.85)
axes[0, 1].axvline(df["log_review_count"].median(), color="orange", linestyle="--", 
                    label=f'중앙값: {df["review_count"].median():.0f}건')
axes[0, 1].set_xlabel("log(1 + 리뷰 수)", fontsize=11)
axes[0, 1].set_ylabel("빈도", fontsize=11)
axes[0, 1].set_title("(b) 리뷰 수 분포 (로그 변환 후)", fontsize=12)
axes[0, 1].legend(fontsize=10)

# (c) 업종 분포
bt_counts = df["business_type"].value_counts()
colors_bt = ["#5B9BD5", "#ED7D31"]
bars = axes[1, 0].bar(bt_counts.index, bt_counts.values, color=colors_bt, edgecolor="white")
for bar, val in zip(bars, bt_counts.values):
    axes[1, 0].text(bar.get_x() + bar.get_width()/2, val + 10, f"{val}\n({val/len(df)*100:.1f}%)", 
                     ha="center", fontsize=10)
axes[1, 0].set_ylabel("가게 수", fontsize=11)
axes[1, 0].set_title("(c) 업종별 가게 수", fontsize=12)

# (d) 행정구별 가게 수
dist_counts = df["district"].value_counts().sort_values()
axes[1, 1].barh(dist_counts.index, dist_counts.values, color="#5B9BD5", edgecolor="white")
for i, val in enumerate(dist_counts.values):
    axes[1, 1].text(val + 2, i, str(val), va="center", fontsize=9)
axes[1, 1].set_xlabel("가게 수", fontsize=11)
axes[1, 1].set_title("(d) 행정구별 가게 수", fontsize=12)

plt.suptitle("Figure 2. 독립변수 분포", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

**해석:**  
- **(a) 가격대:** 중가(MODERATE)가 1,021개(74.8%)로 대다수를 차지한다. 저가 285개(20.9%), 고가 59개(4.3%).
- **(b) 리뷰 수:** 로그 변환 후 비교적 대칭적 분포를 보인다. 원본 중앙값 221건으로, 리뷰가 어느 정도 축적된 가게들이다.
- **(c) 업종:** 음식점(62.3%)이 카페(37.7%)보다 많지만, 두 그룹 모두 충분한 표본 크기를 가진다.
- **(d) 행정구:** 16개 구·군 모두 포함되어 있으며, 최소 51개(연제구)~최대 171개(부산진구)로 분석에 충분하다.

### 3.3 변수 간 상관관계

In [ ]:
# 수치형 변수 간 상관행렬
corr_vars = ["rating", "review_count", "log_review_count", "price_level"]
corr_labels = ["평점", "리뷰 수", "log 리뷰 수", "가격대"]
corr_matrix = df[corr_vars].corr()

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr_matrix, annot=True, fmt=".3f", cmap="RdBu_r", center=0,
            square=True, vmin=-1, vmax=1, ax=ax,
            xticklabels=corr_labels, yticklabels=corr_labels,
            linewidths=0.5)
ax.set_title("Figure 3. 수치형 변수 간 상관행렬 (Pearson)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# 주요 상관계수 출력
print("주요 상관관계:")
print(f"  평점 ↔ 리뷰 수:     r = {corr_matrix.loc['rating','review_count']:.3f}")
print(f"  평점 ↔ log 리뷰 수: r = {corr_matrix.loc['rating','log_review_count']:.3f}")
print(f"  평점 ↔ 가격대:      r = {corr_matrix.loc['rating','price_level']:.3f}")
print(f"  리뷰 수 ↔ 가격대:   r = {corr_matrix.loc['review_count','price_level']:.3f}")

**해석:**  
- 평점과 다른 변수들 간의 상관관계는 전반적으로 **약하다** (|r| < 0.15).
- 평점과 리뷰 수는 약한 음의 상관(r ≈ -0.10)으로, 리뷰가 많을수록 평점이 약간 낮아지는 경향이 있다.
- 이는 리뷰가 많이 쌓일수록 다양한 의견이 반영되어 극단적 고평점이 줄어드는 효과일 수 있다.
- 가격대와 평점 사이에는 거의 상관이 없다 (r ≈ 0.01).
- **→ 개별 변수만으로 평점을 설명하기 어려울 수 있음을 시사한다.**

### 3.4 주요 이변량 관계 탐색

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# (a) 가격대별 평점
order_price = ["저가", "중가", "고가"]
sns.boxplot(data=df, x="price_category", y="rating", order=order_price,
            palette=["#70AD47", "#5B9BD5", "#ED7D31"], ax=axes[0], width=0.5)
# 평균 표시
for i, cat in enumerate(order_price):
    mean_val = df[df["price_category"] == cat]["rating"].mean()
    axes[0].scatter(i, mean_val, color="red", s=60, zorder=5, marker="D")
    axes[0].text(i + 0.15, mean_val, f"{mean_val:.2f}", color="red", fontsize=10, fontweight="bold")
axes[0].set_xlabel("가격대", fontsize=11)
axes[0].set_ylabel("평점", fontsize=11)
axes[0].set_title("(a) 가격대별 평점", fontsize=12)

# (b) 업종별 평점
sns.boxplot(data=df, x="business_type", y="rating",
            palette=["#5B9BD5", "#ED7D31"], ax=axes[1], width=0.4)
for i, bt in enumerate(df["business_type"].unique()):
    mean_val = df[df["business_type"] == bt]["rating"].mean()
    axes[1].scatter(i, mean_val, color="red", s=60, zorder=5, marker="D")
    axes[1].text(i + 0.12, mean_val, f"{mean_val:.2f}", color="red", fontsize=10, fontweight="bold")
axes[1].set_xlabel("업종", fontsize=11)
axes[1].set_ylabel("평점", fontsize=11)
axes[1].set_title("(b) 업종별 평점", fontsize=12)

# (c) 리뷰 수 vs 평점 산점도
axes[2].scatter(df["log_review_count"], df["rating"], alpha=0.25, s=12, color="#5B9BD5")
# 추세선
z = np.polyfit(df["log_review_count"], df["rating"], 1)
p_line = np.poly1d(z)
x_line = np.linspace(df["log_review_count"].min(), df["log_review_count"].max(), 100)
r_val = df["log_review_count"].corr(df["rating"])
axes[2].plot(x_line, p_line(x_line), color="red", linewidth=2, label=f"r = {r_val:.3f}")
axes[2].set_xlabel("log(1 + 리뷰 수)", fontsize=11)
axes[2].set_ylabel("평점", fontsize=11)
axes[2].set_title("(c) 리뷰 수 vs 평점", fontsize=12)
axes[2].legend(fontsize=10)

plt.suptitle("Figure 4. 주요 독립변수와 평점의 관계", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

**해석:**  
- **(a)** 저가, 중가, 고가 간 평점의 중앙값과 평균이 매우 유사하다. 가격대에 따른 뚜렷한 차이는 보이지 않는다.
- **(b)** 음식점과 카페의 평점 분포도 유사하나, 음식점이 약간 더 넓은 분산을 보인다.
- **(c)** 리뷰 수와 평점 사이에 약한 음의 관계가 관찰된다. 리뷰가 적은 가게에서 평점의 변동 폭이 크다.

### 3.5 행정구별 평점 분포

In [ ]:
# 평균 평점 순으로 정렬
district_order = df.groupby("district")["rating"].mean().sort_values(ascending=False).index.tolist()

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# (a) 박스플롯
sns.boxplot(data=df, x="district", y="rating", order=district_order,
            palette="coolwarm_r", ax=axes[0], width=0.6)
axes[0].axhline(df["rating"].mean(), color="red", linestyle="--", alpha=0.5, 
                label=f'전체 평균: {df["rating"].mean():.2f}')
axes[0].set_xlabel("행정구", fontsize=11)
axes[0].set_ylabel("평점", fontsize=11)
axes[0].set_title("(a) 행정구별 평점 분포", fontsize=12)
axes[0].tick_params(axis="x", rotation=45)
axes[0].legend(fontsize=9)

# (b) 평균 + 95% 신뢰구간
dist_stats = df.groupby("district")["rating"].agg(["mean", "std", "count"])
dist_stats["se"] = dist_stats["std"] / np.sqrt(dist_stats["count"])
dist_stats["ci95"] = dist_stats["se"] * 1.96
dist_stats = dist_stats.sort_values("mean", ascending=True)

axes[1].barh(range(len(dist_stats)), dist_stats["mean"],
             xerr=dist_stats["ci95"], color="#5B9BD5", edgecolor="white",
             capsize=3, alpha=0.8)
axes[1].set_yticks(range(len(dist_stats)))
axes[1].set_yticklabels(dist_stats.index)
axes[1].axvline(df["rating"].mean(), color="red", linestyle="--", alpha=0.5,
                label=f'전체 평균: {df["rating"].mean():.2f}')
axes[1].set_xlabel("평균 평점 (95% 신뢰구간)", fontsize=11)
axes[1].set_title("(b) 행정구별 평균 평점과 95% 신뢰구간", fontsize=12)
axes[1].legend(fontsize=9)

plt.suptitle("Figure 5. 행정구별 평점 분포", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# 수치 요약
print("행정구별 요약 (평균 평점 순):")
summary = df.groupby("district")["rating"].agg(["count", "mean", "median", "std"]).round(3)
print(summary.sort_values("mean", ascending=False).to_string())

**해석:**  
- 강서구(4.21), 금정구(4.18), 남구(4.18)이 상대적으로 높은 평균 평점을 보인다.
- 동구(4.02), 사상구(4.06)는 상대적으로 낮은 평균 평점을 보인다.
- 다만 대부분의 행정구가 4.0~4.2 범위 안에 있어 차이가 크지는 않다.
- 95% 신뢰구간이 상당 부분 겹치는 것으로 보아, 통계적 유의성은 별도 검정이 필요하다.

### 3.6 업종 × 가격대 교차 분석

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (a) 업종 × 가격대별 평점
sns.boxplot(data=df, x="price_category", y="rating", hue="business_type",
            order=["저가", "중가", "고가"], palette=["#5B9BD5", "#ED7D31"],
            ax=axes[0], width=0.6)
axes[0].set_xlabel("가격대", fontsize=11)
axes[0].set_ylabel("평점", fontsize=11)
axes[0].set_title("(a) 업종 × 가격대별 평점", fontsize=12)
axes[0].legend(title="업종", fontsize=10)

# (b) 업종 × 리뷰구간별 평점
review_order = ["~50", "51~100", "101~300", "301~1000", "1000+"]
sns.pointplot(data=df, x="review_group", y="rating", hue="business_type",
              order=review_order, palette=["#5B9BD5", "#ED7D31"],
              ax=axes[1], dodge=True, markers=["o", "s"], linestyles=["-", "--"],
              errorbar="ci", capsize=0.1)
axes[1].set_xlabel("리뷰 수 구간", fontsize=11)
axes[1].set_ylabel("평균 평점", fontsize=11)
axes[1].set_title("(b) 업종 × 리뷰 수 구간별 평균 평점", fontsize=12)
axes[1].legend(title="업종", fontsize=10)

plt.suptitle("Figure 6. 업종과 다른 변수의 교호작용 탐색", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

**해석:**  
- **(a)** 저가와 중가에서는 음식점과 카페의 평점이 유사하나, 고가에서는 차이가 나타날 수 있다 (단, 고가 표본이 적어 주의 필요).
- **(b)** 리뷰 수가 적은 구간(~50)에서 카페의 평점이 음식점보다 약간 높으며, 리뷰가 1,000건 이상인 구간에서는 두 업종 모두 평점이 하락하는 경향이 있다.
- **→ 업종에 따라 평점 형성 패턴이 다를 가능성이 있으며, 이는 소주제 4에서 상세 분석한다.**

### 3.7 기술통계 요약표

In [ ]:
# 전체 기술통계
desc = df[["rating", "review_count", "price_level"]].describe().round(3)
desc.index = ["관측치 수", "평균", "표준편차", "최솟값", "Q1(25%)", "중앙값(50%)", "Q3(75%)", "최댓값"]
desc.columns = ["평점(rating)", "리뷰 수(review_count)", "가격대(price_level)"]
print("Table 1. 주요 변수 기술통계량")
print(desc.to_string())

### 3.8 EDA 핵심 요약

| 발견 | 내용 | 시사점 |
|------|------|--------|
| 평점 분포 | 좌편향, 평균 4.14 | 온라인 평점의 긍정 편향 특성 |
| 가격대-평점 | 상관 거의 없음 (r ≈ 0.01) | 비싼 곳이 평점 높지 않음 |
| 리뷰 수-평점 | 약한 음의 상관 (r ≈ -0.10) | 리뷰 많을수록 평점 약간 하락 |
| 행정구별 | 시각적 차이 존재 | 통계 검정 필요 |
| 업종별 | 유사하나 교호작용 가능성 | 소주제 4에서 상세 분석 |

---
**다음 단계:** 소주제별 통계 분석 (Part 2에서 진행)

# Part 2: 소주제별 분석
## 소주제 1: 가격대(저가/중가/고가)에 따라 평점에 유의한 차이가 있는가?

**가설:**
- H₀: 가격대별 평점의 분포에 차이가 없다
- H₁: 적어도 하나의 가격대에서 평점의 분포가 다르다

### 1.1 가격대별 기술통계

### 1.2 가격대별 평점 분포 시각화

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
order = ["저가", "중가", "고가"]
colors = ["#70AD47", "#5B9BD5", "#ED7D31"]

# (a) 박스플롯
sns.boxplot(data=df, x="price_category", y="rating", order=order,
            palette=colors, ax=axes[0], width=0.5)
# 평균 마커
for i, cat in enumerate(order):
    mean_val = price_groups[cat].mean()
    axes[0].scatter(i, mean_val, color="red", s=80, zorder=5, marker="D", edgecolors="white", linewidth=1)
    axes[0].text(i + 0.2, mean_val, f"{mean_val:.3f}", color="red", fontsize=10, fontweight="bold")
axes[0].set_xlabel("가격대", fontsize=11)
axes[0].set_ylabel("평점", fontsize=11)
axes[0].set_title("(a) 박스플롯", fontsize=12)

# (b) 바이올린 플롯
sns.violinplot(data=df, x="price_category", y="rating", order=order,
               palette=colors, ax=axes[1], inner="quartile")
axes[1].set_xlabel("가격대", fontsize=11)
axes[1].set_ylabel("평점", fontsize=11)
axes[1].set_title("(b) 바이올린 플롯", fontsize=12)

# (c) 히스토그램 (겹쳐서)
for cat, color in zip(order, colors):
    axes[2].hist(price_groups[cat], bins=20, alpha=0.5, color=color, label=cat, density=True, edgecolor="white")
axes[2].set_xlabel("평점", fontsize=11)
axes[2].set_ylabel("밀도", fontsize=11)
axes[2].set_title("(c) 가격대별 평점 밀도", fontsize=12)
axes[2].legend()

plt.suptitle("Figure 7. 소주제 1 — 가격대별 평점 분포", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### 1.3 검정 가정 확인

ANOVA를 적용하려면 **(1) 정규성**과 **(2) 등분산성** 가정이 필요하다.  
가정이 충족되지 않으면 비모수 검정(Kruskal-Wallis)을 사용한다.

In [ ]:
# ── (1) 정규성 검정: Shapiro-Wilk ──
print("(1) 정규성 검정 (Shapiro-Wilk)")
print(f"    H₀: 해당 그룹의 평점은 정규분포를 따른다")
print(f"    α = 0.05\n")

normality_results = {}
for cat in order:
    g = price_groups[cat]
    # 표본이 5000개 초과하면 샘플링
    sample = g.sample(min(500, len(g)), random_state=42) if len(g) > 500 else g
    stat, p = shapiro(sample)
    is_normal = p >= 0.05
    normality_results[cat] = is_normal
    print(f"    {cat} (n={len(g)}): W = {stat:.4f}, p = {p:.6f} → {'정규분포 ✓' if is_normal else '비정규 ✗'}")

print(f"\n    결론: 저가, 중가가 비정규 → ANOVA 가정 불충족")

# ── (2) 등분산성 검정: Levene ──
lev_stat, lev_p = levene(*price_groups.values())
print(f"\n(2) 등분산성 검정 (Levene)")
print(f"    H₀: 세 그룹의 분산이 동일하다")
print(f"    통계량 = {lev_stat:.4f}, p = {lev_p:.6f}")
print(f"    → {'등분산 ✓' if lev_p >= 0.05 else '이분산 ✗ (등분산 가정 불충족)'}")

# 시각화: 정규성 Q-Q plot
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, (cat, color) in enumerate(zip(order, colors)):
    from scipy.stats import probplot
    probplot(price_groups[cat], dist="norm", plot=axes[i])
    axes[i].set_title(f"{cat} Q-Q Plot (n={len(price_groups[cat])})", fontsize=11)
    axes[i].get_lines()[0].set_color(color)
plt.suptitle("Figure 8. 가격대별 평점 정규성 확인 (Q-Q Plot)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"\n※ 정규성과 등분산성 모두 충족되지 않으므로,")
print(f"   비모수 검정인 Kruskal-Wallis 검정을 주 검정으로 사용한다.")
print(f"   ANOVA는 참고용으로 함께 보고한다.")

### 1.4 통계 검정

- **주 검정:** Kruskal-Wallis 검정 (비모수, 정규성·등분산성 가정 불요)
- **참고:** One-way ANOVA (모수 검정)

In [ ]:
# ── Kruskal-Wallis 검정 ──
kw_stat, kw_p = kruskal(*price_groups.values())

print("Kruskal-Wallis 검정")
print(f"  H₀: 세 가격대의 평점 분포가 동일하다")
print(f"  H₁: 적어도 하나의 가격대에서 평점 분포가 다르다")
print(f"  α = 0.05")
print(f"")
print(f"  검정통계량 H = {kw_stat:.4f}")
print(f"  p-value     = {kw_p:.6f}")
print(f"  결론: p = {kw_p:.4f} {'< 0.05 → H₀ 기각. 가격대별 평점에 유의한 차이가 있다.' if kw_p < 0.05 else '>= 0.05 → H₀ 기각 불가.'}")

# ── 효과 크기 (η² 근사) ──
# η² = (H - k + 1) / (N - k), k=그룹 수
n_total = len(df)
k = 3
eta_sq = (kw_stat - k + 1) / (n_total - k)
print(f"\n  효과 크기 (η² ≈ {eta_sq:.4f})")
print(f"  → {'작은 효과 (η² < 0.06)' if eta_sq < 0.06 else ('중간 효과' if eta_sq < 0.14 else '큰 효과')}")
print(f"  → 통계적으로 유의하지만 실질적 차이는 크지 않음")

# ── ANOVA (참고) ──
f_stat, f_p = f_oneway(*price_groups.values())
print(f"\n[참고] One-way ANOVA")
print(f"  F = {f_stat:.4f}, p = {f_p:.6f}")
print(f"  → {'유의함' if f_p < 0.05 else '유의하지 않음'} (정규성·등분산 가정 미충족이므로 참고용)")

### 1.5 사후검정

Kruskal-Wallis에서 유의한 차이가 확인되었으므로,  
**어떤 그룹 쌍**에서 차이가 있는지 Mann-Whitney U 검정으로 확인한다.  
다중비교에 따른 Type I 오류 증가를 보정하기 위해 **Bonferroni 보정**을 적용한다.

In [ ]:
print("사후검정: Mann-Whitney U (Bonferroni 보정)")
print(f"  비교 횟수: 3회 → Bonferroni 보정 적용 (α' = 0.05 / 3 = 0.0167)")
print()

pairs = [("저가", "중가"), ("저가", "고가"), ("중가", "고가")]
posthoc_results = []

for g1, g2 in pairs:
    stat, p = mannwhitneyu(price_groups[g1], price_groups[g2], alternative="two-sided")
    p_adj = min(p * len(pairs), 1.0)  # Bonferroni 보정
    
    # 효과 크기 (rank-biserial correlation)
    n1, n2 = len(price_groups[g1]), len(price_groups[g2])
    r_effect = 1 - (2 * stat) / (n1 * n2)
    
    sig = "***" if p_adj < 0.001 else ("**" if p_adj < 0.01 else ("*" if p_adj < 0.05 else "n.s."))
    
    posthoc_results.append({
        "비교": f"{g1} vs {g2}",
        "U": f"{stat:.0f}",
        "p (원본)": f"{p:.6f}",
        "p (보정)": f"{p_adj:.6f}",
        "r (효과크기)": f"{r_effect:.4f}",
        "판정": sig,
    })
    
    print(f"  {g1} vs {g2}:")
    print(f"    U = {stat:.0f}, p = {p:.6f}, p_adj = {p_adj:.6f} {sig}")
    print(f"    효과 크기 r = {r_effect:.4f} ({'작은' if abs(r_effect) < 0.3 else ('중간' if abs(r_effect) < 0.5 else '큰')} 효과)")
    print()

posthoc_df = pd.DataFrame(posthoc_results)
print("\nTable 3. 사후검정 결과 요약")
print(posthoc_df.to_string(index=False))

### 1.6 사후검정 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# (a) 그룹 간 비교 시각화 (평균 + 95% CI)
means = [price_groups[cat].mean() for cat in order]
sems = [price_groups[cat].sem() for cat in order]
ci95 = [sem * 1.96 for sem in sems]

axes[0].bar(order, means, yerr=ci95, color=colors, edgecolor="white", 
            capsize=8, alpha=0.8, width=0.5)
# 유의한 차이 표시
# 저가 vs 중가가 유의
y_max = max(means) + max(ci95) + 0.02
axes[0].plot([0, 1], [y_max, y_max], color="black", linewidth=1.5)
axes[0].text(0.5, y_max + 0.005, "*", ha="center", fontsize=16, fontweight="bold")

for i, (m, c) in enumerate(zip(means, ci95)):
    axes[0].text(i, m - c - 0.025, f"{m:.3f}", ha="center", fontsize=10, fontweight="bold")

axes[0].set_ylabel("평균 평점", fontsize=11)
axes[0].set_title("(a) 가격대별 평균 평점 (95% CI)\n* p < 0.05 (Bonferroni 보정)", fontsize=12)
axes[0].set_ylim(3.9, 4.35)

# (b) 각 그룹 평점 분포 (strip + box)
sns.stripplot(data=df, x="price_category", y="rating", order=order,
              palette=colors, alpha=0.2, size=3, jitter=0.2, ax=axes[1])
sns.boxplot(data=df, x="price_category", y="rating", order=order,
            palette=colors, ax=axes[1], width=0.3, 
            boxprops=dict(alpha=0.6), showfliers=False)
axes[1].set_xlabel("가격대", fontsize=11)
axes[1].set_ylabel("평점", fontsize=11)
axes[1].set_title("(b) 가격대별 개별 데이터 + 박스플롯", fontsize=12)

plt.suptitle("Figure 9. 소주제 1 — 가격대별 평점 비교", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### 1.7 소주제 1 결론

| 항목 | 결과 |
|------|------|
| 검정 방법 | Kruskal-Wallis (비모수) |
| 검정통계량 | H = 7.85 |
| p-value | 0.0197 |
| 결론 | **가격대별 평점에 통계적으로 유의한 차이가 있다** (p < 0.05) |
| 사후검정 | **저가 > 중가** (p_adj = 0.017), 나머지 쌍은 유의하지 않음 |
| 효과 크기 | η² ≈ 0.004 (매우 작은 효과) |

**해석:**
- 통계적으로 유의한 차이는 존재하지만, 효과 크기(η² ≈ 0.004)가 매우 작다.
- 이는 가격대 간 평점 차이가 **실질적으로는 미미**하다는 것을 의미한다.
- 구체적으로, **저가 음식점/카페의 평점(4.178)이 중가(4.133)보다 오히려 약간 높다.**
- "비싼 곳이 평점이 높다"는 직관과 달리, 가격은 평점의 주요 결정 요인이 아님을 시사한다.
- 이는 평점이 음식의 맛, 서비스 등 가격 이외의 **직접 경험적 요인**에 더 크게 좌우됨을 의미할 수 있다.

**한계점:**
- price_level은 Google이 매기는 상대적 등급이며, 실제 메뉴 가격이 아님
- 고가 그룹(59개)의 표본이 상대적으로 작아, 고가 관련 비교의 검정력이 제한적
- class imbalance (중가 74.8%)가 존재하므로 해석에 주의 필요

## 소주제 2: 리뷰 수가 많을수록 평점이 높거나 안정적인가?

이 소주제는 두 가지 질문을 포함한다:
1. 리뷰 수와 평점 사이에 **방향성**(양/음의 상관)이 있는가?
2. 리뷰 수가 많을수록 평점이 **안정적**(분산이 작아지는가)인가?

두 번째 질문은 통계학의 **대수의 법칙(Law of Large Numbers)** 관점에서  
리뷰(표본)가 축적될수록 평점(표본평균)의 변동성이 줄어드는지를 실증적으로 확인하는 것이다.

### 2.1 리뷰 수와 평점의 상관분석

In [ ]:
from scipy.stats import pearsonr, spearmanr, levene, kruskal, mannwhitneyu

# ── Pearson / Spearman 상관계수 ──
r_pearson, p_pearson = pearsonr(df["review_count"], df["rating"])
r_spearman, p_spearman = spearmanr(df["review_count"], df["rating"])
r_log_pearson, p_log_pearson = pearsonr(df["log_review_count"], df["rating"])

print("상관분석 결과")
print(f"{'─'*60}")
print(f"{'방법':<25} {'상관계수':<12} {'p-value':<15} {'판정'}")
print(f"{'─'*60}")
print(f"{'Pearson (원본)':<25} {r_pearson:<12.4f} {p_pearson:<15.6f} {'*' if p_pearson < 0.05 else 'n.s.'}")
print(f"{'Spearman (순위)':<25} {r_spearman:<12.4f} {p_spearman:<15.6f} {'***' if p_spearman < 0.001 else '*' if p_spearman < 0.05 else 'n.s.'}")
print(f"{'Pearson (log 변환)':<25} {r_log_pearson:<12.4f} {p_log_pearson:<15.6f} {'*' if p_log_pearson < 0.05 else 'n.s.'}")
print(f"{'─'*60}")
print(f"\n해석:")
print(f"  - 세 방법 모두 약한 음의 상관이 유의하게 나타남 (p < 0.05)")
print(f"  - Spearman ρ = {r_spearman:.4f}로 가장 강한 관계 → 비선형 단조 관계 존재")
print(f"  - 방향: 리뷰 수가 많을수록 평점이 약간 낮아지는 경향")

### 2.2 산점도 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (a) 원본 스케일
axes[0].scatter(df["review_count"], df["rating"], alpha=0.2, s=10, color="#5B9BD5")
axes[0].set_xlabel("리뷰 수", fontsize=11)
axes[0].set_ylabel("평점", fontsize=11)
axes[0].set_title(f"(a) 리뷰 수 vs 평점 (원본)\nPearson r = {r_pearson:.4f}", fontsize=12)
axes[0].set_xlim(0, df["review_count"].quantile(0.98))

# (b) 로그 스케일 + 추세선
axes[1].scatter(df["log_review_count"], df["rating"], alpha=0.2, s=10, color="#5B9BD5")
# 추세선
z = np.polyfit(df["log_review_count"], df["rating"], 1)
p_line = np.poly1d(z)
x_line = np.linspace(df["log_review_count"].min(), df["log_review_count"].max(), 100)
axes[1].plot(x_line, p_line(x_line), color="red", linewidth=2, 
             label=f"추세선 (기울기={z[0]:.4f})")
axes[1].set_xlabel("log(1 + 리뷰 수)", fontsize=11)
axes[1].set_ylabel("평점", fontsize=11)
axes[1].set_title(f"(b) log 리뷰 수 vs 평점\nSpearman ρ = {r_spearman:.4f} (p < 0.001)", fontsize=12)
axes[1].legend(fontsize=10)

plt.suptitle("Figure 10. 소주제 2 — 리뷰 수와 평점의 관계", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

**해석:**  
- 산점도에서 뚜렷한 선형 관계는 보이지 않으나, **리뷰 수가 적은 영역에서 평점의 분산이 크다.**
- 리뷰가 많은 가게들은 평점이 4.0 부근에 밀집되어 있다.
- 이 패턴은 다음 섹션에서 구간별로 더 명확하게 확인한다.

### 2.3 리뷰 수 구간별 평점 분석

리뷰 수를 5개 구간으로 나누어 평점의 **수준(평균)**과 **변동성(표준편차)**을 비교한다.

In [ ]:
# 구간별 기술통계
review_order = ["~50", "51~100", "101~300", "301~1000", "1000+"]

grp_stats = df.groupby("review_group", observed=True)["rating"].agg(
    ["count", "mean", "median", "std"]
).reindex(review_order).round(3)
grp_stats.columns = ["n", "평균", "중앙값", "표준편차"]

print("Table 4. 리뷰 수 구간별 평점 기술통계량")
print(grp_stats.to_string())

print(f"\n핵심 관찰:")
print(f"  평균: {grp_stats['평균'].iloc[0]:.3f} (~50건) → {grp_stats['평균'].iloc[-1]:.3f} (1000+건)")
print(f"  표준편차: {grp_stats['표준편차'].iloc[0]:.3f} (~50건) → {grp_stats['표준편차'].iloc[-1]:.3f} (1000+건)")
print(f"  → 리뷰 수 증가에 따라 평점이 소폭 하락하고, 표준편차가 크게 감소")

### 2.4 구간별 시각화 — 평점 수준과 분산

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# (a) 구간별 박스플롯
sns.boxplot(data=df, x="review_group", y="rating", order=review_order,
            palette="viridis", ax=axes[0], width=0.6)
axes[0].set_xlabel("리뷰 수 구간", fontsize=11)
axes[0].set_ylabel("평점", fontsize=11)
axes[0].set_title("(a) 구간별 평점 분포", fontsize=12)

# (b) 구간별 평균 + 95% CI
means = grp_stats["평균"]
stds = grp_stats["표준편차"]
ns = grp_stats["n"]
sems = stds / np.sqrt(ns)
ci95 = sems * 1.96

axes[1].errorbar(range(len(review_order)), means, yerr=ci95, 
                 fmt="o-", color="#5B9BD5", linewidth=2, markersize=8, capsize=5)
axes[1].set_xticks(range(len(review_order)))
axes[1].set_xticklabels(review_order)
axes[1].set_xlabel("리뷰 수 구간", fontsize=11)
axes[1].set_ylabel("평균 평점 (95% CI)", fontsize=11)
axes[1].set_title("(b) 구간별 평균 평점 추이", fontsize=12)

# (c) 구간별 표준편차 (분산 비교) — 핵심!
colors_var = plt.cm.Reds(np.linspace(0.3, 0.8, len(review_order)))
bars = axes[2].bar(review_order, stds, color=colors_var, edgecolor="white")
for bar, val in zip(bars, stds):
    axes[2].text(bar.get_x() + bar.get_width()/2, val + 0.005, f"{val:.3f}", 
                 ha="center", fontsize=10, fontweight="bold")
axes[2].set_xlabel("리뷰 수 구간", fontsize=11)
axes[2].set_ylabel("평점 표준편차", fontsize=11)
axes[2].set_title("(c) 구간별 평점 표준편차\n→ 리뷰 많을수록 분산 감소", fontsize=12)

plt.suptitle("Figure 11. 소주제 2 — 리뷰 수 구간별 평점 수준과 분산", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### 2.5 통계 검정

In [ ]:
# ── (1) 구간별 평점 수준 차이: Kruskal-Wallis ──
groups_data = [df[df["review_group"] == g]["rating"].dropna() for g in review_order]

kw_stat, kw_p = kruskal(*groups_data)
print("(1) 구간별 평점 수준 차이 (Kruskal-Wallis)")
print(f"  H₀: 리뷰 수 구간에 따라 평점 분포에 차이가 없다")
print(f"  H = {kw_stat:.4f}, p = {kw_p:.6f}")
print(f"  → {'유의한 차이 있음' if kw_p < 0.05 else '유의하지 않음'} (α = 0.05)")

# 효과 크기
n_total = len(df)
eta_sq = (kw_stat - len(review_order) + 1) / (n_total - len(review_order))
print(f"  효과 크기 η² ≈ {eta_sq:.4f} ({'작은' if eta_sq < 0.06 else '중간'} 효과)")

# ── (2) 구간별 분산 차이: Levene 검정 ──
lev_stat, lev_p = levene(*groups_data)
print(f"\n(2) 구간별 평점 분산 차이 (Levene)")
print(f"  H₀: 리뷰 수 구간에 따라 평점의 분산이 동일하다")
print(f"  F = {lev_stat:.4f}, p = {lev_p:.6f}")
print(f"  → {'분산에 유의한 차이 있음' if lev_p < 0.05 else '유의하지 않음'} (α = 0.05)")

# ── (3) 사후 비교: 인접 구간끼리 ──
print(f"\n(3) 인접 구간 사후비교 (Mann-Whitney U)")
for i in range(len(review_order) - 1):
    g1_name = review_order[i]
    g2_name = review_order[i + 1]
    g1 = df[df["review_group"] == g1_name]["rating"]
    g2 = df[df["review_group"] == g2_name]["rating"]
    stat, p = mannwhitneyu(g1, g2, alternative="two-sided")
    sig = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "n.s."))
    print(f"  {g1_name} vs {g2_name}: U={stat:.0f}, p={p:.4f} {sig}  (평균: {g1.mean():.3f} vs {g2.mean():.3f})")

### 2.6 대수의 법칙 관점 시각화

통계학의 대수의 법칙에 의하면, 표본 크기(리뷰 수)가 증가하면  
표본평균(평점)의 변동성이 감소하여 모평균에 수렴한다.  
이를 실증적으로 확인한다.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# 리뷰 수별 평점 산점도 (로그 스케일)
scatter = ax.scatter(df["log_review_count"], df["rating"], 
                     alpha=0.15, s=10, color="#5B9BD5", label="개별 가게")

# 구간별 평균 + 표준편차 범위
midpoints = []
for g in review_order:
    sub = df[df["review_group"] == g]
    mid_log = sub["log_review_count"].median()
    midpoints.append(mid_log)

grp_means = [df[df["review_group"] == g]["rating"].mean() for g in review_order]
grp_stds = [df[df["review_group"] == g]["rating"].std() for g in review_order]

ax.errorbar(midpoints, grp_means, yerr=grp_stds, fmt="D-", color="red", 
            linewidth=2.5, markersize=10, capsize=6, capthick=2,
            label="구간 평균 ± 1 SD", zorder=5)

# 전체 평균선
ax.axhline(df["rating"].mean(), color="gray", linestyle=":", alpha=0.7,
           label=f'전체 평균 ({df["rating"].mean():.2f})')

ax.set_xlabel("log(1 + 리뷰 수)", fontsize=12)
ax.set_ylabel("평점", fontsize=12)
ax.set_title("Figure 12. 리뷰 수 증가에 따른 평점의 수렴 (대수의 법칙)\n"
             "→ 리뷰가 쌓일수록 평점이 전체 평균 근처로 수렴하고 변동성이 감소",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10, loc="lower left")

plt.tight_layout()
plt.show()

### 2.7 소주제 2 결론

| 항목 | 결과 |
|------|------|
| 상관관계 | Spearman ρ = -0.135 (p < 0.001), 약한 음의 상관 |
| 구간별 평점 차이 | Kruskal-Wallis H = 27.72, p < 0.001 → 유의 |
| 구간별 분산 차이 | Levene F = 46.33, p < 0.001 → 유의 |
| 효과 크기 | η² ≈ 0.017 (작은 효과) |

**해석:**

1. **평점 수준:** 리뷰 수가 많을수록 평점이 **소폭 하락**하는 경향이 있다  
   (~50건: 4.151 → 1000+건: 4.092). 이는 리뷰가 적은 가게에서  
   소수의 긍정적 리뷰만 있을 때 평점이 과대 추정되는 효과로 해석된다.

2. **평점 안정성:** 리뷰 수가 많을수록 평점의 표준편차가 **크게 감소**한다  
   (~50건: 0.499 → 1000+건: 0.240). 이는 **대수의 법칙**과 일치하는  
   결과로, 리뷰(표본)가 축적될수록 평점(표본평균)이 안정화됨을 실증적으로 보여준다.

3. **실용적 시사점:** 리뷰 수가 매우 적은 가게(~50건)의 평점은 변동성이 커서  
   신뢰도가 낮을 수 있으며, 최소 100건 이상의 리뷰가 있어야 평점이 안정적이라 할 수 있다.

## 소주제 3: 부산 내 행정구별로 평점 분포에 차이가 있는가?

부산광역시는 15개 구와 1개 군(기장군)으로 구성되어 있다.  
이 소주제에서는 행정구(위치)에 따라 카페/음식점의 평점 분포가 달라지는지 분석한다.

**가설:**
- H₀: 16개 행정구의 평점 분포가 모두 동일하다
- H₁: 적어도 하나의 행정구에서 평점 분포가 다르다

### 3.1 행정구별 기술통계

In [ ]:
from scipy.stats import kruskal, mannwhitneyu, shapiro, levene, f_oneway

# 행정구별 기술통계 (평균 순 정렬)
district_stats = df.groupby("district")["rating"].agg(
    ["count", "mean", "median", "std"]
).round(3).sort_values("mean", ascending=False)
district_stats.columns = ["n", "평균", "중앙값", "표준편차"]

print("Table 5. 행정구별 평점 기술통계량 (평균 내림차순)")
print(district_stats.to_string())

print(f"\n전체 평균: {df['rating'].mean():.3f}")
print(f"행정구 간 평균의 범위: {district_stats['평균'].min():.3f} ~ {district_stats['평균'].max():.3f}")
print(f"행정구 간 평균의 차이: {district_stats['평균'].max() - district_stats['평균'].min():.3f}")

**해석:**
- 평균 평점이 가장 높은 구는 **강서구(4.212)**, 가장 낮은 구는 **동구(4.022)**이다.
- 16개 행정구의 평균 평점 범위는 4.022~4.212로, **차이가 0.190**에 불과하다.
- 이는 부산 전체적으로 음식점/카페 평점이 비교적 **균질**함을 시사한다.
- 표본 크기는 최소 51개(연제구)~최대 171개(부산진구)로 분석에 충분하다.

### 3.2 행정구별 평점 분포 시각화

In [ ]:
district_order = district_stats.index.tolist()  # 평균 순

fig, axes = plt.subplots(2, 1, figsize=(15, 12))

# (a) 박스플롯
sns.boxplot(data=df, x="district", y="rating", order=district_order,
            palette="coolwarm_r", ax=axes[0], width=0.6)
axes[0].axhline(df["rating"].mean(), color="red", linestyle="--", alpha=0.6,
                label=f'전체 평균: {df["rating"].mean():.3f}')
# 각 구 평균 마커
for i, d in enumerate(district_order):
    mean_val = df[df["district"] == d]["rating"].mean()
    axes[0].scatter(i, mean_val, color="red", s=40, zorder=5, marker="D")
axes[0].set_xlabel("")
axes[0].set_ylabel("평점", fontsize=11)
axes[0].set_title("(a) 행정구별 평점 분포 (박스플롯, 평균 순)", fontsize=12)
axes[0].tick_params(axis="x", rotation=45)
axes[0].legend(fontsize=10)

# (b) 평균 + 95% 신뢰구간
dist_ci = district_stats.copy()
dist_ci["se"] = dist_ci["표준편차"] / np.sqrt(dist_ci["n"])
dist_ci["ci95"] = dist_ci["se"] * 1.96
dist_ci_sorted = dist_ci.sort_values("평균", ascending=True)

colors_bar = plt.cm.coolwarm(np.linspace(0.2, 0.8, len(dist_ci_sorted)))
axes[1].barh(range(len(dist_ci_sorted)), dist_ci_sorted["평균"],
             xerr=dist_ci_sorted["ci95"], color=colors_bar, edgecolor="white",
             capsize=4, alpha=0.85)
axes[1].set_yticks(range(len(dist_ci_sorted)))
axes[1].set_yticklabels([f"{d} (n={int(dist_ci_sorted.loc[d, 'n'])})" for d in dist_ci_sorted.index])
axes[1].axvline(df["rating"].mean(), color="red", linestyle="--", alpha=0.6,
                label=f'전체 평균: {df["rating"].mean():.3f}')
axes[1].set_xlabel("평균 평점", fontsize=11)
axes[1].set_title("(b) 행정구별 평균 평점과 95% 신뢰구간", fontsize=12)
axes[1].legend(fontsize=10)

plt.suptitle("Figure 13. 소주제 3 — 행정구별 평점 분포", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

**해석:**
- **(a) 박스플롯:** 행정구별 중앙값(4.1~4.2)과 사분위 범위가 대부분 겹치고 있어,  
  시각적으로 큰 차이를 확인하기 어렵다. 다만 동구, 사상구는 다른 구에 비해  
  분산(상자 크기)이 큰 편이다.
- **(b) 신뢰구간:** 대부분의 행정구에서 95% 신뢰구간이 **전체 평균(4.142)을 포함**하고 있다.  
  강서구와 동구만 신뢰구간이 전체 평균에서 약간 벗어나 있으나,  
  다른 구들의 신뢰구간과는 여전히 상당 부분 겹친다.

### 3.3 검정 가정 확인

In [ ]:
# ── 정규성 ──
print("정규성 검정 (Shapiro-Wilk, 각 행정구)")
non_normal = 0
for d in district_order:
    g = df[df["district"] == d]["rating"]
    sample = g.sample(min(500, len(g)), random_state=42) if len(g) > 500 else g
    stat, p = shapiro(sample)
    is_normal = p >= 0.05
    if not is_normal:
        non_normal += 1
    print(f"  {d:<6s} (n={len(g):>3d}): W={stat:.4f}, p={p:.4f} → {'정규 ✓' if is_normal else '비정규 ✗'}")

print(f"\n  비정규 분포 행정구: {non_normal}/16개")
print(f"  → 과반수가 비정규이므로 비모수 검정(Kruskal-Wallis) 사용")

# ── 등분산성 ──
district_groups = [df[df["district"] == d]["rating"].values for d in district_order]
lev_stat, lev_p = levene(*district_groups)
print(f"\n등분산성 검정 (Levene)")
print(f"  F = {lev_stat:.4f}, p = {lev_p:.6f}")
print(f"  → {'등분산 ✓' if lev_p >= 0.05 else '이분산 ✗'}")

### 3.4 통계 검정

In [ ]:
# ── Kruskal-Wallis 검정 ──
kw_stat, kw_p = kruskal(*district_groups)

print("Kruskal-Wallis 검정")
print(f"  H₀: 16개 행정구의 평점 분포가 모두 동일하다")
print(f"  H₁: 적어도 하나의 행정구에서 평점 분포가 다르다")
print(f"  α = 0.05")
print(f"")
print(f"  검정통계량 H = {kw_stat:.4f}")
print(f"  p-value     = {kw_p:.6f}")

if kw_p < 0.05:
    print(f"  결론: p = {kw_p:.4f} < 0.05 → H₀ 기각. 행정구별 평점에 유의한 차이가 있다.")
else:
    print(f"  결론: p = {kw_p:.4f} ≥ 0.05 → H₀ 기각 불가.")
    print(f"         16개 행정구 전체를 동시에 비교했을 때,")
    print(f"         통계적으로 유의한 차이를 발견할 수 없다.")

# 효과 크기
n_total = len(df)
k = 16
eta_sq = (kw_stat - k + 1) / (n_total - k)
print(f"\n  효과 크기 η² ≈ {eta_sq:.4f} ({'작은 효과' if eta_sq < 0.06 else '중간 효과'})")

# ── ANOVA (참고) ──
f_stat, f_p = f_oneway(*district_groups)
print(f"\n[참고] One-way ANOVA: F = {f_stat:.4f}, p = {f_p:.6f}")

**해석:**

Kruskal-Wallis 검정 결과 p = 0.087로, **유의수준 0.05에서 H₀를 기각할 수 없다.**  
즉, 16개 행정구의 평점 분포가 **전체적으로 동일하다**는 귀무가설을 유지한다.

이 결과는 다음과 같이 해석할 수 있다:
- 부산 지역 카페/음식점의 평점은 행정구에 관계없이 **비교적 균질**하다.
- 특정 행정구가 "맛집 구"이거나 "평점이 낮은 구"라고 단정짓기 어렵다.
- 16개 그룹을 동시에 비교하면 검정의 보수성이 높아져,  
  실제로 존재할 수 있는 **부분적 차이**를 탐지하지 못했을 가능성도 있다.

따라서 아래에서 **상위 그룹 vs 하위 그룹**을 나누어 추가 분석한다.

### 3.5 추가 분석: 상위 행정구 vs 하위 행정구

In [ ]:
# 평균 평점 기준 상위 5개 / 하위 5개
top5 = district_stats.head(5).index.tolist()
bot5 = district_stats.tail(5).index.tolist()

top5_ratings = df[df["district"].isin(top5)]["rating"]
bot5_ratings = df[df["district"].isin(bot5)]["rating"]

stat_tb, p_tb = mannwhitneyu(top5_ratings, bot5_ratings, alternative="two-sided")

# 효과 크기 (rank-biserial)
n1, n2 = len(top5_ratings), len(bot5_ratings)
r_effect = 1 - (2 * stat_tb) / (n1 * n2)

print("추가 분석: 상위 5개구 vs 하위 5개구")
print(f"  상위 5개구: {', '.join(top5)}")
print(f"    n = {len(top5_ratings)}, 평균 = {top5_ratings.mean():.3f}, 중앙값 = {top5_ratings.median()}")
print(f"  하위 5개구: {', '.join(bot5)}")
print(f"    n = {len(bot5_ratings)}, 평균 = {bot5_ratings.mean():.3f}, 중앙값 = {bot5_ratings.median()}")
print(f"")
print(f"  Mann-Whitney U = {stat_tb:.0f}")
print(f"  p-value = {p_tb:.6f}")
print(f"  효과 크기 r = {r_effect:.4f}")
print(f"  → {'유의한 차이 있음' if p_tb < 0.05 else '유의하지 않음'}")

if p_tb < 0.05:
    print(f"\n  해석:")
    print(f"    16개 행정구 전체 비교에서는 유의하지 않았으나,")
    print(f"    평균 평점 상위 5개구와 하위 5개구를 비교하면 유의한 차이가 나타난다.")
    print(f"    상위 5개구의 평균({top5_ratings.mean():.3f})이 하위 5개구({bot5_ratings.mean():.3f})보다")
    print(f"    {top5_ratings.mean() - bot5_ratings.mean():.3f}점 높다.")

### 3.6 추가 분석 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# (a) 상위 5 vs 하위 5 비교
df_topbot = df.copy()
df_topbot["group"] = df_topbot["district"].apply(
    lambda x: "상위 5개구" if x in top5 else ("하위 5개구" if x in bot5 else "중간")
)
df_topbot_filtered = df_topbot[df_topbot["group"] != "중간"]

sns.boxplot(data=df_topbot_filtered, x="group", y="rating", 
            order=["상위 5개구", "하위 5개구"],
            palette=["#5B9BD5", "#ED7D31"], ax=axes[0], width=0.4)

for i, grp in enumerate(["상위 5개구", "하위 5개구"]):
    mean_val = df_topbot_filtered[df_topbot_filtered["group"] == grp]["rating"].mean()
    axes[0].scatter(i, mean_val, color="red", s=80, zorder=5, marker="D", edgecolors="white")
    axes[0].text(i + 0.15, mean_val, f"{mean_val:.3f}", color="red", fontsize=11, fontweight="bold")

# 유의성 표시
y_max = 4.6
if p_tb < 0.05:
    axes[0].plot([0, 1], [y_max, y_max], color="black", linewidth=1.5)
    sig_mark = "***" if p_tb < 0.001 else ("**" if p_tb < 0.01 else "*")
    axes[0].text(0.5, y_max + 0.02, sig_mark, ha="center", fontsize=14, fontweight="bold")

axes[0].set_ylabel("평점", fontsize=11)
axes[0].set_title(f"(a) 상위 5개구 vs 하위 5개구\np = {p_tb:.6f}", fontsize=12)

# (b) 행정구별 평균 (상위/하위 색 구분)
dist_mean_sorted = district_stats.sort_values("평균", ascending=True)
colors_district = []
for d in dist_mean_sorted.index:
    if d in top5:
        colors_district.append("#5B9BD5")
    elif d in bot5:
        colors_district.append("#ED7D31")
    else:
        colors_district.append("#AAAAAA")

axes[1].barh(range(len(dist_mean_sorted)), dist_mean_sorted["평균"],
             color=colors_district, edgecolor="white", alpha=0.85)
axes[1].set_yticks(range(len(dist_mean_sorted)))
axes[1].set_yticklabels(dist_mean_sorted.index)
axes[1].axvline(df["rating"].mean(), color="red", linestyle="--", alpha=0.5,
                label=f'전체 평균: {df["rating"].mean():.3f}')

# 범례
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor="#5B9BD5", label="상위 5개구"),
                   Patch(facecolor="#ED7D31", label="하위 5개구"),
                   Patch(facecolor="#AAAAAA", label="중간")]
axes[1].legend(handles=legend_elements, fontsize=9, loc="lower right")
axes[1].set_xlabel("평균 평점", fontsize=11)
axes[1].set_title("(b) 행정구별 평균 평점 (상위/하위 구분)", fontsize=12)

plt.suptitle("Figure 14. 소주제 3 — 행정구 그룹별 평점 비교", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### 3.7 업종별 행정구 패턴 (탐색적)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for i, bt in enumerate(["음식점", "카페"]):
    sub = df[df["business_type"] == bt]
    d_mean = sub.groupby("district")["rating"].mean().sort_values(ascending=True)
    
    colors_bt = ["#5B9BD5" if d in top5 else ("#ED7D31" if d in bot5 else "#AAAAAA") for d in d_mean.index]
    axes[i].barh(range(len(d_mean)), d_mean.values, color=colors_bt, edgecolor="white", alpha=0.85)
    axes[i].set_yticks(range(len(d_mean)))
    axes[i].set_yticklabels(d_mean.index)
    axes[i].axvline(sub["rating"].mean(), color="red", linestyle="--", alpha=0.5)
    axes[i].set_xlabel("평균 평점", fontsize=11)
    axes[i].set_title(f"{bt} (n={len(sub)})", fontsize=12)

plt.suptitle("Figure 15. 업종별 행정구 평균 평점 비교", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

**해석:**
- 음식점과 카페의 행정구별 순위가 **완전히 일치하지는 않는다.**
- 예를 들어, 음식점에서 평점이 높은 구가 카페에서는 중간 수준일 수 있다.
- 이는 행정구의 특성(관광지, 주거지, 상업지구 등)이  
  업종에 따라 **다르게 작용**할 수 있음을 시사하며, 소주제 4(업종별 분석)에서 더 살펴본다.

### 3.8 소주제 3 결론

| 항목 | 결과 |
|------|------|
| 검정 방법 | Kruskal-Wallis (비모수) |
| 검정통계량 | H = 22.86 |
| p-value | 0.087 |
| 결론 (전체) | 16개 행정구 전체 비교 시 **유의한 차이 없음** (p ≥ 0.05) |
| 추가 분석 | 상위 5개구 vs 하위 5개구 비교 시 **유의한 차이 있음** (p < 0.001) |
| 효과 크기 | 작은 효과 |

**해석:**

1. **전체 비교(16개 구 동시):** 유의한 차이가 발견되지 않았다 (p = 0.087).  
   이는 부산 지역의 카페/음식점 평점이 행정구에 관계없이 **비교적 균질**함을 의미한다.  
   즉, 특정 행정구가 "맛집 동네"이거나 "평점이 낮은 동네"라고 단정짓기 어렵다.

2. **부분 비교(상위 vs 하위):** 평균 평점 상위 5개구(강서, 금정, 남, 영도, 해운대)와  
   하위 5개구(기장, 동래, 사상, 동, 북)를 비교하면 **유의한 차이**가 나타난다.  
   이는 극단적인 그룹 간에는 차이가 존재하지만, 전체 16개 구를 동시에 보면  
   중간 그룹들이 차이를 희석시키기 때문으로 해석된다.

3. **실용적 시사점:** 부산에서 카페/음식점의 평점은 위치(행정구)보다는  
   개별 가게의 특성(맛, 서비스 등)에 더 크게 좌우되는 것으로 보인다.  
   다만, 동구·사상구 등 상대적으로 평점이 낮은 지역이 존재하며,  
   이는 해당 지역의 상권 특성이나 고객층의 차이에 기인할 수 있다.

4. **방법론적 시사점:** Kruskal-Wallis는 16개 그룹을 동시에 비교하므로  
   보수적(conservative)인 결과를 낸다. p = 0.087은 유의수준 0.10에서는  
   유의하며, 표본이 더 크거나 그룹 수가 적었다면 유의했을 가능성이 있다.  
   "유의하지 않다"는 것이 "차이가 없다"를 **증명하는 것은 아님**에 유의해야 한다.

## 소주제 4: 업종(카페 vs 음식점)에 따라 평점 형성 패턴이 다른가?

카페와 음식점은 소비자가 평가하는 기준이 다를 수 있다.  
카페는 분위기·인테리어가, 음식점은 맛·양이 더 중요할 수 있으며,  
이에 따라 가격·리뷰 수가 평점에 미치는 영향도 업종별로 다를 수 있다.

이 소주제에서는 세 가지를 분석한다:
1. 업종 간 평점 수준 차이
2. 업종별로 리뷰 수-평점 관계가 다른가
3. 업종별로 가격대-평점 관계가 다른가 (교호작용)

### 4.1 업종별 기술통계

In [ ]:
from scipy.stats import mannwhitneyu, spearmanr, kruskal, levene

rest = df[df["business_type"] == "음식점"]
cafe = df[df["business_type"] == "카페"]

# 기술통계 비교
desc_bt = pd.DataFrame({
    "음식점": {
        "n": len(rest),
        "평점 평균": f"{rest['rating'].mean():.3f}",
        "평점 중앙값": f"{rest['rating'].median():.1f}",
        "평점 표준편차": f"{rest['rating'].std():.3f}",
        "리뷰 수 중앙값": f"{rest['review_count'].median():.0f}",
        "리뷰 수 평균": f"{rest['review_count'].mean():.0f}",
    },
    "카페": {
        "n": len(cafe),
        "평점 평균": f"{cafe['rating'].mean():.3f}",
        "평점 중앙값": f"{cafe['rating'].median():.1f}",
        "평점 표준편차": f"{cafe['rating'].std():.3f}",
        "리뷰 수 중앙값": f"{cafe['review_count'].median():.0f}",
        "리뷰 수 평균": f"{cafe['review_count'].mean():.0f}",
    },
})

print("Table 6. 업종별 기술통계량 비교")
print(desc_bt.to_string())

**해석:**
- 카페의 평균 평점(4.192)이 음식점(4.112)보다 **0.080점 높다.**
- 카페의 표준편차(0.358)가 음식점(0.282)보다 크다 → 카페는 평점의 변동이 더 크다.
- 이는 카페가 "매우 좋거나 매우 나쁜" 극단적 평가를 더 많이 받는다는 의미일 수 있다.

### 4.2 업종별 평점 분포 시각화

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
colors_bt = ["#5B9BD5", "#ED7D31"]

# (a) 박스플롯
sns.boxplot(data=df, x="business_type", y="rating", palette=colors_bt, ax=axes[0], width=0.4)
for i, bt in enumerate(["음식점", "카페"]):
    mean_val = df[df["business_type"]==bt]["rating"].mean()
    axes[0].scatter(i, mean_val, color="red", s=80, zorder=5, marker="D", edgecolors="white")
    axes[0].text(i + 0.15, mean_val, f"{mean_val:.3f}", color="red", fontsize=11, fontweight="bold")
axes[0].set_xlabel("업종", fontsize=11)
axes[0].set_ylabel("평점", fontsize=11)
axes[0].set_title("(a) 업종별 평점 분포", fontsize=12)

# (b) 겹친 히스토그램
axes[1].hist(rest["rating"], bins=25, alpha=0.6, color=colors_bt[0], label="음식점", density=True, edgecolor="white")
axes[1].hist(cafe["rating"], bins=25, alpha=0.6, color=colors_bt[1], label="카페", density=True, edgecolor="white")
axes[1].axvline(rest["rating"].mean(), color=colors_bt[0], linestyle="--", linewidth=2)
axes[1].axvline(cafe["rating"].mean(), color=colors_bt[1], linestyle="--", linewidth=2)
axes[1].set_xlabel("평점", fontsize=11)
axes[1].set_ylabel("밀도", fontsize=11)
axes[1].set_title("(b) 업종별 평점 밀도 비교", fontsize=12)
axes[1].legend(fontsize=10)

# (c) 바이올린 + strip
sns.violinplot(data=df, x="business_type", y="rating", palette=colors_bt, 
               ax=axes[2], inner="quartile", alpha=0.7)
axes[2].set_xlabel("업종", fontsize=11)
axes[2].set_ylabel("평점", fontsize=11)
axes[2].set_title("(c) 바이올린 플롯", fontsize=12)

plt.suptitle("Figure 16. 소주제 4 — 업종별 평점 분포", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

**해석:**
- **(a)** 카페의 중앙값(4.2)이 음식점(4.1)보다 높고, 박스 위치가 전반적으로 위에 있다.
- **(b)** 음식점은 4.0~4.2에 밀집된 좁은 분포, 카페는 4.0~4.4에 걸친 넓은 분포를 보인다.  
  특히 카페는 4.5 이상의 고평점 비율이 음식점보다 높다.
- **(c)** 바이올린 플롯에서 카페의 분포가 더 넓고, 상단(고평점)으로 더 뻗어있음을 확인할 수 있다.

### 4.3 업종 간 평점 차이 검정

In [ ]:
# Mann-Whitney U 검정
stat, p = mannwhitneyu(rest["rating"], cafe["rating"], alternative="two-sided")
n1, n2 = len(rest), len(cafe)
r_effect = 1 - (2 * stat) / (n1 * n2)

print("업종 간 평점 차이 검정 (Mann-Whitney U)")
print(f"  H₀: 음식점과 카페의 평점 분포가 동일하다")
print(f"  H₁: 음식점과 카페의 평점 분포가 다르다")
print(f"  α = 0.05")
print(f"")
print(f"  U = {stat:.0f}")
print(f"  p-value = {p:.6f}")
print(f"  효과 크기 r = {r_effect:.4f}")
print(f"")
print(f"  결론: p < 0.001 → H₀ 기각.")
print(f"  카페(평균 {cafe['rating'].mean():.3f})의 평점이")
print(f"  음식점(평균 {rest['rating'].mean():.3f})보다 통계적으로 유의하게 높다.")
print(f"")
print(f"  효과 크기 r = {r_effect:.4f}는 {'작은' if abs(r_effect) < 0.3 else '중간'} 수준이다.")

# 분산 비교 (Levene)
lev_stat, lev_p = levene(rest["rating"], cafe["rating"])
print(f"\n분산 비교 (Levene)")
print(f"  F = {lev_stat:.4f}, p = {lev_p:.6f}")
print(f"  → {'분산에 유의한 차이 있음' if lev_p < 0.05 else '분산 차이 없음'}")
if lev_p < 0.05:
    print(f"  → 카페(σ={cafe['rating'].std():.3f})의 평점 분산이")
    print(f"    음식점(σ={rest['rating'].std():.3f})보다 유의하게 크다.")
    print(f"    이는 카페에 대한 평가가 음식점보다 더 양극화되어 있음을 의미한다.")

### 4.4 업종별 리뷰 수-평점 관계 비교

리뷰 수가 평점에 미치는 영향이 업종에 따라 다른지 확인한다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, (bt, color, sub) in enumerate(zip(["음식점", "카페"], colors_bt, [rest, cafe])):
    r_sp, p_sp = spearmanr(sub["log_review_count"], sub["rating"])
    
    axes[i].scatter(sub["log_review_count"], sub["rating"], alpha=0.2, s=10, color=color)
    
    # 추세선
    z = np.polyfit(sub["log_review_count"], sub["rating"], 1)
    p_line = np.poly1d(z)
    x_line = np.linspace(sub["log_review_count"].min(), sub["log_review_count"].max(), 100)
    axes[i].plot(x_line, p_line(x_line), color="red", linewidth=2)
    
    axes[i].set_xlabel("log(1 + 리뷰 수)", fontsize=11)
    axes[i].set_ylabel("평점", fontsize=11)
    sig = "***" if p_sp < 0.001 else ("*" if p_sp < 0.05 else "n.s.")
    axes[i].set_title(f"{bt} (n={len(sub)})\nSpearman ρ = {r_sp:.4f} (p = {p_sp:.4f}) {sig}", fontsize=12)

plt.suptitle("Figure 17. 업종별 리뷰 수-평점 관계", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# 수치 비교
print("업종별 리뷰 수-평점 Spearman 상관")
for bt, sub in [("음식점", rest), ("카페", cafe)]:
    r_sp, p_sp = spearmanr(sub["log_review_count"], sub["rating"])
    sig = "***" if p_sp < 0.001 else ("*" if p_sp < 0.05 else "n.s.")
    print(f"  {bt}: ρ = {r_sp:.4f}, p = {p_sp:.6f} {sig}")

print(f"\n해석:")
print(f"  음식점: ρ = -0.119 (유의) → 리뷰 많을수록 평점 약간 하락")
print(f"  카페:   ρ = -0.082 (유의하지 않음) → 리뷰 수와 평점 관련 약함")
print(f"  → 음식점에서 리뷰 축적 효과(평점 하락)가 카페보다 더 뚜렷하다.")
print(f"     이는 음식점이 더 다양한 고객층(관광객 포함)으로부터")
print(f"     리뷰를 받아 평점이 '평균으로 회귀'하는 경향이 강하기 때문일 수 있다.")

### 4.5 업종별 가격대-평점 관계 비교 (교호작용)

가격대가 평점에 미치는 영향이 업종에 따라 다른지(교호작용) 확인한다.

In [ ]:
# 업종 × 가격대 교차 기술통계
cross_stats = df.groupby(["business_type", "price_category"])["rating"].agg(
    ["count", "mean", "std"]
).round(3)
cross_stats.columns = ["n", "평균", "표준편차"]

print("Table 7. 업종 × 가격대별 평점")
print(cross_stats.to_string())

# 업종별 가격대 효과 검정
print(f"\n업종별 가격대 효과 (Kruskal-Wallis)")
for bt in ["음식점", "카페"]:
    sub = df[df["business_type"] == bt]
    groups = [sub[sub["price_category"]==c]["rating"] for c in ["저가","중가","고가"]]
    H, p = kruskal(*groups)
    sig = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "n.s."))
    print(f"  {bt}: H = {H:.4f}, p = {p:.4f} {sig}")

### 4.6 교호작용 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
price_order = ["저가", "중가", "고가"]

# (a) 교호작용 플롯 (Point plot)
sns.pointplot(data=df, x="price_category", y="rating", hue="business_type",
              order=price_order, palette=colors_bt, ax=axes[0],
              dodge=0.2, markers=["o", "s"], linestyles=["-", "--"],
              errorbar="ci", capsize=0.1)
axes[0].set_xlabel("가격대", fontsize=11)
axes[0].set_ylabel("평균 평점", fontsize=11)
axes[0].set_title("(a) 업종 × 가격대 교호작용\n→ 선이 평행하지 않으면 교호작용 존재", fontsize=12)
axes[0].legend(title="업종", fontsize=10)

# (b) 업종 × 가격대 박스플롯
sns.boxplot(data=df, x="price_category", y="rating", hue="business_type",
            order=price_order, palette=colors_bt, ax=axes[1], width=0.6)
axes[1].set_xlabel("가격대", fontsize=11)
axes[1].set_ylabel("평점", fontsize=11)
axes[1].set_title("(b) 업종 × 가격대별 평점 분포", fontsize=12)
axes[1].legend(title="업종", fontsize=10)

plt.suptitle("Figure 18. 소주제 4 — 업종 × 가격대 교호작용", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("해석:")
print("  음식점: 저가(4.157) > 중가(4.101) ≈ 고가(4.094)")
print("    → 저가 음식점의 평점이 가장 높음 (가성비 효과)")
print("    → 가격대별 유의한 차이 있음 (p = 0.027)")
print("")
print("  카페: 고가(4.273) > 저가(4.207) ≈ 중가(4.185)")
print("    → 고가 카페의 평점이 가장 높음 (프리미엄 효과)")
print("    → 가격대별 유의한 차이 없음 (p = 0.207)")
print("")
print("  ★ 교호작용 패턴:")
print("    음식점은 '저가가 높은' 패턴 (가성비 선호)")
print("    카페는 '고가가 높은' 패턴 (프리미엄 선호)")
print("    → 소비자의 가격-품질 기대가 업종에 따라 다르다!")

### 4.7 업종별 리뷰 구간-평점 패턴 비교

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
review_order = ["~50", "51~100", "101~300", "301~1000", "1000+"]

sns.pointplot(data=df, x="review_group", y="rating", hue="business_type",
              order=review_order, palette=colors_bt, ax=ax,
              dodge=0.15, markers=["o", "s"], linestyles=["-", "--"],
              errorbar="ci", capsize=0.1)
ax.set_xlabel("리뷰 수 구간", fontsize=11)
ax.set_ylabel("평균 평점 (95% CI)", fontsize=11)
ax.set_title("Figure 19. 업종별 리뷰 수 구간에 따른 평점 추이", fontsize=13, fontweight="bold")
ax.legend(title="업종", fontsize=10)
plt.tight_layout()
plt.show()

print("해석:")
print("  두 업종 모두 리뷰 수가 증가하면 평점이 하락하는 추세를 보인다.")
print("  그러나 카페는 모든 리뷰 구간에서 음식점보다 평점이 높으며,")
print("  특히 리뷰가 적은 구간(~50건)에서 카페의 평점이 음식점보다 훨씬 높다.")
print("  리뷰 1000건 이상에서는 두 업종의 평점이 수렴하는 경향이 있다.")
print("")
print("  이는 소규모(리뷰 적은) 카페가 특히 높은 평점을 받는 경향이 있음을 시사하며,")
print("  '숨은 카페' 효과 — 단골 중심의 소수 리뷰어가 높은 점수를 주는 현상 — 로 해석할 수 있다.")

### 4.8 소주제 4 결론

| 항목 | 결과 |
|------|------|
| 업종 간 평점 차이 | Mann-Whitney p < 0.001, **카페(4.192) > 음식점(4.112)** |
| 효과 크기 | r = 0.165 (작은 효과) |
| 업종 간 분산 차이 | Levene p < 0.001, **카페의 평점 분산이 더 크다** |
| 리뷰-평점 관계 | 음식점: ρ=-0.119 (유의) / 카페: ρ=-0.082 (비유의) |
| 가격대-평점 관계 | 음식점: 저가>중가≈고가 (유의) / 카페: 차이 없음 |
| 교호작용 | **존재함** — 음식점은 가성비, 카페는 프리미엄 패턴 |

**해석:**

1. **카페의 평점이 음식점보다 유의하게 높다.**  
   이는 카페와 음식점의 **평가 기준 차이**에 기인할 수 있다.  
   카페는 분위기·인테리어·음료 품질이 복합적으로 평가되며,  
   이런 요소들은 상대적으로 **일정한 품질**을 유지하기 쉽다.  
   반면 음식점은 맛·양·가격·서비스 등 **다차원적 기준**으로 평가되어  
   부정적 평가를 받을 가능성이 더 높다.

2. **카페의 평점 분산이 더 크다.**  
   카페에 대한 평가가 더 **양극화**되어 있다는 뜻이다.  
   "인스타 감성"에 특화된 카페는 극찬을,  
   그렇지 못한 카페는 혹평을 받는 경향이 있을 수 있다.

3. **가격대-평점 관계가 업종별로 다르다 (교호작용).**  
   음식점에서는 **저가의 평점이 가장 높다** (가성비 효과).  
   소비자는 저렴한 음식점에 대해 기대치가 낮고, 그 기대를 초과하면 높게 평가한다.  
   카페에서는 오히려 **고가의 평점이 높은 경향**이 있다 (프리미엄 효과).  
   이는 고가 카페가 분위기·품질 면에서 확실한 만족을 제공하기 때문으로 해석된다.

4. **리뷰 축적 효과도 업종별로 다르다.**  
   음식점은 리뷰가 많을수록 평점이 유의하게 하락하지만,  
   카페는 그 효과가 약하다. 이는 음식점이 더 다양한 고객층에 노출되어  
   평점이 "평균으로 회귀"하는 반면, 카페는 비교적 일정한 고객층이  
   방문하기 때문일 수 있다.

## 소주제 5: 가격, 리뷰 수, 지역, 업종을 종합하면 평점을 얼마나 설명할 수 있으며, 그 한계는 무엇인가?

소주제 1~4에서 개별 요인과 평점의 관계를 확인하였다.  
이제 모든 요인을 **동시에** 투입하는 다중회귀 모델을 구축하고,  
그 **설명력(R²)**, **각 변수의 기여도**, **모델의 한계**를 분석한다.

**모델:**
$$\text{rating}_i = \beta_0 + \beta_1 \cdot \log(\text{review}) + \beta_2 \cdot \text{price} + \beta_3 \cdot \text{업종} + \sum_{k} \gamma_k \cdot \text{district}_k + \epsilon_i$$

### 5.1 모델링 준비

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# ── 특성 변수 준비 ──
# 수치형: log_review_count, price_level
# 범주형 → 더미변수: district (기준: 강서구), business_type (기준: 음식점)
X = pd.get_dummies(
    df[["log_review_count", "price_level", "district", "business_type"]],
    columns=["district", "business_type"],
    drop_first=True,
)
y = df["rating"]

print(f"독립변수: {X.shape[1]}개")
print(f"  수치형 (2개): log_review_count, price_level")
print(f"  더미변수 (16개): district 15개 (기준: 강서구), business_type 1개 (기준: 음식점)")
print(f"종속변수: rating (n = {len(y)}, 평균 = {y.mean():.3f}, σ = {y.std():.3f})")

### 5.2 다중공선성 진단 (VIF)

회귀 모델에서 독립변수 간 강한 상관이 있으면 계수 추정이 불안정해진다.  
VIF(Variance Inflation Factor)로 이를 진단한다.

$$VIF_j = \frac{1}{1 - R_j^2}$$

일반적으로 VIF > 10이면 심각한 다중공선성을 의미한다.

In [ ]:
def calc_vif(X_df):
    """VIF 수동 계산"""
    vif_data = []
    for col in X_df.columns:
        y_temp = X_df[col]
        X_temp = X_df.drop(columns=[col])
        r2 = LinearRegression().fit(X_temp, y_temp).score(X_temp, y_temp)
        vif = 1 / (1 - r2) if r2 < 1 else float("inf")
        vif_data.append({"변수": col, "VIF": round(vif, 2)})
    return pd.DataFrame(vif_data)

# 주요 수치형 + 업종 변수
X_vif = df[["log_review_count", "price_level"]].copy()
X_vif["business_type_카페"] = (df["business_type"] == "카페").astype(int)

vif_result = calc_vif(X_vif)
print("Table 8. 다중공선성 진단 (VIF)")
print(vif_result.to_string(index=False))
print(f"\n판단: 모든 VIF < 2 → 다중공선성 문제 없음")

### 5.3 모델 적합

In [ ]:
# Train/Test 분리
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"학습: {len(X_train)}개 (80%), 테스트: {len(X_test)}개 (20%)")

# 다중회귀 적합
model = LinearRegression()
model.fit(X_train, y_train)

# 예측
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# 성능 지표
r2_train = r2_score(y_train, y_pred_train)
r2_test = r2_score(y_test, y_pred_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
mae_test = mean_absolute_error(y_test, y_pred_test)

print(f"\n다중회귀 모델 성능")
print(f"{'─'*40}")
print(f"  R² (학습):  {r2_train:.4f}")
print(f"  R² (테스트): {r2_test:.4f}")
print(f"  RMSE:       {rmse_test:.4f}")
print(f"  MAE:        {mae_test:.4f}")
print(f"{'─'*40}")
print(f"\n해석:")
print(f"  R² = {r2_test:.4f}는 가격, 리뷰 수, 지역, 업종이")
print(f"  평점 변동의 약 {r2_test*100:.1f}%만 설명한다는 의미이다.")
print(f"  나머지 {(1-r2_test)*100:.1f}%는 이 변수들로 설명할 수 없다.")

### 5.4 회귀 계수 분석

In [ ]:
# 회귀 계수 정리
coef_df = pd.DataFrame({
    "변수": X.columns,
    "계수(β)": model.coef_,
    "|계수|": np.abs(model.coef_),
}).sort_values("|계수|", ascending=False)

print(f"절편 (β₀) = {model.intercept_:.4f}")
print(f"\nTable 9. 다중회귀 계수 (절대값 순)")
print(coef_df.to_string(index=False))

### 5.5 회귀 계수 시각화 + 해석

In [ ]:
top_n = 10
top_coefs = coef_df.head(top_n).sort_values("계수(β)")

fig, ax = plt.subplots(figsize=(10, 6))
colors_coef = ["#C44E52" if v < 0 else "#5B9BD5" for v in top_coefs["계수(β)"]]
ax.barh(range(len(top_coefs)), top_coefs["계수(β)"].values, color=colors_coef, edgecolor="white")
ax.set_yticks(range(len(top_coefs)))
ax.set_yticklabels(top_coefs["변수"].values)
ax.axvline(0, color="black", linewidth=0.8)

for i, v in enumerate(top_coefs["계수(β)"].values):
    offset = 0.003 if v >= 0 else -0.003
    ha = "left" if v >= 0 else "right"
    ax.text(v + offset, i, f"{v:.4f}", va="center", ha=ha, fontsize=9, fontweight="bold")

ax.set_xlabel("회귀 계수 (β)", fontsize=11)
ax.set_title("Figure 20. 다중회귀 계수 (상위 10개)\n파랑 = 평점에 양(+)의 영향 / 빨강 = 음(-)의 영향", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# 해석
print("회귀 계수 해석:")

lr_coef = coef_df[coef_df["변수"]=="log_review_count"]["계수(β)"].values[0]
print(f"\n  1) log_review_count (β = {lr_coef:.4f})")
print(f"     리뷰 수가 증가하면 평점이 소폭 {'감소' if lr_coef < 0 else '증가'}하는 경향.")
print(f"     소주제 2의 결과(리뷰 축적 시 평점의 평균 회귀)와 일치한다.")

pl_coef = coef_df[coef_df["변수"]=="price_level"]["계수(β)"].values[0]
print(f"\n  2) price_level (β = {pl_coef:.4f})")
print(f"     가격대가 1단계 올라갈 때 평점 변화: {pl_coef:.4f}점.")
print(f"     사실상 무시 가능한 수준 → 소주제 1(효과 크기 매우 작음)과 일치한다.")

bt_cols = [c for c in coef_df["변수"] if "business_type" in c]
if bt_cols:
    bt_coef = coef_df[coef_df["변수"]==bt_cols[0]]["계수(β)"].values[0]
    print(f"\n  3) {bt_cols[0]} (β = {bt_coef:.4f})")
    print(f"     다른 변수를 통제한 상태에서도 카페가 음식점보다 약 {bt_coef:.3f}점 높다.")
    print(f"     소주제 4의 결과(카페 > 음식점)와 일치한다.")

dist_coefs = coef_df[coef_df["변수"].str.startswith("district_")].sort_values("계수(β)", ascending=False)
print(f"\n  4) district (행정구)")
print(f"     기준(강서구) 대비 평점이 높은 구: {', '.join(dist_coefs[dist_coefs['계수(β)']>0]['변수'].str.replace('district_','').head(3))}")
print(f"     기준(강서구) 대비 평점이 낮은 구: {', '.join(dist_coefs[dist_coefs['계수(β)']<0]['변수'].str.replace('district_','').tail(3))}")
print(f"     행정구 간 계수 범위: {dist_coefs['계수(β)'].min():.4f} ~ {dist_coefs['계수(β)'].max():.4f}")
print(f"     → 행정구 효과도 크지 않음. 소주제 3(전체 비교 시 유의하지 않음)과 일치한다.")

### 5.6 잔차 진단

다중회귀의 4가지 가정을 잔차를 통해 진단한다:
1. **선형성**: 잔차에 패턴이 없어야 함
2. **정규성**: 잔차가 정규분포를 따라야 함
3. **등분산성**: 잔차의 분산이 일정해야 함
4. **독립성**: 잔차 간 자기상관이 없어야 함

In [ ]:
residuals = y_test.values - y_pred_test

fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# (a) 잔차 vs 예측값 → 선형성 + 등분산성
axes[0,0].scatter(y_pred_test, residuals, alpha=0.25, s=12, color="#5B9BD5")
axes[0,0].axhline(0, color="red", linestyle="--", linewidth=1)
axes[0,0].set_xlabel("예측값", fontsize=11)
axes[0,0].set_ylabel("잔차", fontsize=11)
axes[0,0].set_title("(a) 잔차 vs 예측값\n→ 패턴 없으면 선형성·등분산성 충족", fontsize=11)

# (b) 잔차 히스토그램 → 정규성
axes[0,1].hist(residuals, bins=30, color="#5B9BD5", edgecolor="white", density=True, alpha=0.7)
from scipy.stats import norm
x_norm = np.linspace(residuals.min(), residuals.max(), 100)
axes[0,1].plot(x_norm, norm.pdf(x_norm, residuals.mean(), residuals.std()),
               "r-", linewidth=2, label="정규분포")
axes[0,1].set_xlabel("잔차", fontsize=11)
axes[0,1].set_ylabel("밀도", fontsize=11)
axes[0,1].set_title("(b) 잔차 분포\n→ 종형이면 정규성 충족", fontsize=11)
axes[0,1].legend()

# (c) Q-Q Plot → 정규성 정밀 확인
from scipy.stats import probplot
probplot(residuals, dist="norm", plot=axes[1,0])
axes[1,0].set_title("(c) Q-Q Plot\n→ 대각선에 가까우면 정규분포", fontsize=11)

# (d) 실제값 vs 예측값
axes[1,1].scatter(y_test, y_pred_test, alpha=0.25, s=12, color="#5B9BD5")
lims = [min(y_test.min(), y_pred_test.min())-0.1, max(y_test.max(), y_pred_test.max())+0.1]
axes[1,1].plot(lims, lims, "r--", linewidth=1.5, label="y = x (완벽한 예측)")
axes[1,1].set_xlabel("실제 평점", fontsize=11)
axes[1,1].set_ylabel("예측 평점", fontsize=11)
axes[1,1].set_title("(d) 실제 vs 예측\n→ 대각선에 가까울수록 정확", fontsize=11)
axes[1,1].legend(fontsize=9)

plt.suptitle("Figure 21. 다중회귀 잔차 진단", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# 잔차 통계
from scipy.stats import shapiro
res_sample = residuals[:500] if len(residuals) > 500 else residuals
res_shapiro = shapiro(res_sample)

print("잔차 진단 요약")
print(f"{'─'*55}")
print(f"  잔차 평균:     {residuals.mean():.4f} (0에 가까우면 ✓)")
print(f"  잔차 표준편차:  {residuals.std():.4f}")
print(f"  잔차 왜도:     {pd.Series(residuals).skew():.4f}")
print(f"  Shapiro-Wilk:  W={res_shapiro[0]:.4f}, p={res_shapiro[1]:.6f}")
print(f"{'─'*55}")
print(f"\n  (a) 선형성·등분산성: 뚜렷한 패턴 없음 → 크게 위배되지 않음")
print(f"      단, 예측값이 {y_pred_test.min():.2f}~{y_pred_test.max():.2f}에 밀집")
print(f"      → 모델이 대부분 비슷한 값을 예측하고 있다는 뜻")
print(f"  (b) 정규성: 좌편향 존재, Shapiro p < 0.05 → 완전한 정규분포는 아님")
print(f"  (c) Q-Q Plot: 양 끝단 이탈 → 극단적 평점(2~3점대, 5점)에서 예측 부정확")
print(f"  (d) 실제 vs 예측: 예측이 좁은 범위에 집중 → 실제 평점의 다양성 미반영")

### 5.7 설명력의 한계: 왜 R²가 낮은가?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

explained = r2_test * 100
unexplained = (1 - r2_test) * 100

wedges, texts, autotexts = ax.pie(
    [explained, unexplained],
    labels=None,
    autopct="%1.1f%%",
    colors=["#5B9BD5", "#E0E0E0"],
    startangle=90,
    wedgeprops=dict(edgecolor="white", linewidth=2),
    textprops=dict(fontsize=14, fontweight="bold"),
)

ax.legend(
    [f"설명 가능 ({explained:.1f}%)\n가격, 리뷰 수, 지역, 업종",
     f"설명 불가 ({unexplained:.1f}%)\n맛, 서비스, 청결도, 분위기 등"],
    loc="center left", bbox_to_anchor=(1, 0.5), fontsize=11,
)

ax.set_title("Figure 22. 평점 변동의 설명력 분해\n→ 외부 관찰 가능 요인으로 평점의 약 4%만 설명 가능",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print("R²가 낮은 이유:")
print(f"")
print(f"  1) 평점은 '직접 경험'의 결과물이다.")
print(f"     맛, 서비스, 청결도, 분위기 등 실제 방문 경험이 평점을 결정한다.")
print(f"     이런 요인은 가격·위치·리뷰 수로 포착할 수 없다.")
print(f"")
print(f"  2) 평점의 변동 범위가 좁다.")
print(f"     표준편차 {y.std():.3f} → 설명할 변동 자체가 작다.")
print(f"")
print(f"  3) 소주제 1~4의 개별 효과가 모두 작았다.")
print(f"     개별 효과가 작으면, 합쳐도 설명력이 낮을 수밖에 없다.")
print(f"")
print(f"  4) 이 결과 자체가 의미 있는 발견이다.")
print(f"     '비싸면 맛있다', '해운대가 맛집이 많다' 같은 통념이")
print(f"     데이터로 지지되지 않음을 실증적으로 보여주었다.")

### 5.8 소주제 5 결론

| 항목 | 결과 |
|------|------|
| 모델 | 다중회귀 (OLS) |
| 독립변수 | log_review_count, price_level, district(15), business_type(1) |
| R² (테스트) | **0.037** |
| RMSE | 0.334 |
| VIF | 모두 < 2 (다중공선성 없음) |
| 가장 큰 계수 | business_type_카페 (+), 일부 district |

**종합 해석:**

가격, 리뷰 수, 지역, 업종을 모두 투입한 다중회귀 모델의 R²는 약 **0.037**로,  
이 변수들이 평점 변동의 약 **3.7%만 설명**한다.

이는 모델의 실패가 아니라, **평점의 본질적 특성**을 반영하는 결과이다.  
평점은 가격·위치·인지도 같은 외부 조건이 아니라,  
**맛·서비스·분위기** 같은 직접 경험적 요인에 의해 결정된다.

소비자에게 실질적으로 중요한 것은 "어디에 있는 얼마짜리 가게"가 아니라  
**"그 가게가 제공하는 고유한 경험의 품질"**이다.

이 결론은 소주제 1~4의 분석 결과(개별 효과 크기가 모두 작았음)와 **일관**되며,  
대주제의 질문 "이러한 요인들의 설명력은 어디까지인가?"에 대한 답이 된다.

**한계점 및 향후 연구:**
- 맛, 서비스, 청결도 등 직접 경험적 변수가 포함되지 않았다.
- Google priceLevel은 상대적 등급이며 실제 메뉴 가격이 아니다.
- 향후 텍스트 리뷰의 감성 분석(Sentiment Analysis)을 추가하면  
  직접 경험적 요인을 간접적으로 포착하여 설명력을 높일 수 있을 것이다.